# STEP33 — Virtual Gene Knockout Analysis (UPDATED FROZEN DESIGN)

## Objective

This notebook prepares the frozen framework for virtual gene perturbation analysis.

Scientific question:

> Under the frozen MS3 proliferative/mitotic high-risk malignant program, can controlled virtual perturbation of key regulators reveal program dependency?

This notebook intentionally stops before biological simulation.

Current phase:

**Audit → Environment Lock → Input Preparation**

No knockout simulation is performed in this version.


In [ ]:
import psutil

mem = psutil.virtual_memory()

print("Total RAM GB:", round(mem.total / 1024**3, 2))
print("Available RAM GB:", round(mem.available / 1024**3, 2))
print("Used %:", mem.percent)

In [ ]:
import psutil

mem = psutil.virtual_memory()

print("Total RAM GB:", round(mem.total / 1024**3, 2))
print("Available RAM GB:", round(mem.available / 1024**3, 2))
print("Used %:", mem.percent)

# STEP33 Frozen Principle

Workflow:

Frozen malignant state  
↓  
Frozen GRN architecture  
↓  
Controlled virtual perturbation  
↓  
Endpoint evaluation

Firewall:

- No new clustering
- No malignant state redefinition
- No GRN rediscovery
- No TF reselection
- No Original RiskScore reconstruction


# STEP33 Frozen Inputs

## scRNA reference

Dataset:
GSE149614

Frozen malignant population:
12,057 cells

States:
MS1-MS8


## GRN

Source:
Step29 frozen motif-filtered regulatory network

Original topology:

32,530 edges

Self-loops:

140

CellOracle-compatible simulation topology:

32,390 edges

Only self-loops are removed.
No other topology modification is allowed.


# STEP33 Environment Lock

Required kernel:

Python (Step33 CellOracle)

Environment:

ko33_celloracle_wsl

Expected:

Python 3.10.21

CellOracle 0.18.0

Scanpy 1.9.8

AnnData 0.10.8

numpy 1.26.4

pandas 1.5.3


In [ ]:
# STEP33 environment verification

import sys

print("Python:", sys.version)

try:
    import celloracle as co
    print("CellOracle:", co.__version__)
except Exception as e:
    print("CellOracle import failed:", e)

try:
    import scanpy as sc
    print("Scanpy:", sc.__version__)
except Exception as e:
    print("Scanpy import failed:", e)

try:
    import anndata as ad
    print("AnnData:", ad.__version__)
except Exception as e:
    print("AnnData import failed:", e)

print("STEP33 environment check completed")


# STEP33A — Frozen Input Audit

Purpose:

Verify that all frozen inputs required for virtual perturbation are available.

Checks:

- malignant cell correspondence
- raw count availability
- frozen GRN availability
- target gene availability

Frozen candidate regulators:

Primary:

- FOXM1
- MYBL2
- E2F8
- TFDP1
- E2F1

Reference:

- BRCA1

No biological simulation starts before this audit passes.


# STEP33B — Historical CellOracle Compatibility Audit

Status:

COMPLETED (historical record)

Finding:

Direct use of the original 32,530-edge topology is incompatible with CellOracle 0.18.0 because self-loop handling occurs during model fitting.

This record documents a computational compatibility test only.

No virtual knockout was performed.
No biological interpretation was made.


# STEP33B-R — Frozen Topology Adaptation Lock

Decision:

Create a CellOracle-compatible simulation topology.

Rules:

Allowed:
- remove self-loops only

Forbidden:
- remove non-self edges
- add new edges
- infer new GRN

Topology:

Original:
32,530 edges

Removed:
140 self-loops

Simulation:
32,390 edges

This topology is the only permitted input for downstream simulation.


# STEP33C — CellOracle Reconstruction

NOT STARTED

Future step:

Construct CellOracle simulation object using:

- frozen malignant raw counts
- frozen MS annotations
- frozen GRN-compatible topology

No biological perturbation code is included in this notebook version.


# STEP33 Current Status

STEP33A:
READY

STEP33B0:
COMPLETED

STEP33B1:
ARCHIVED COMPATIBILITY RECORD

STEP33B-R:
LOCKED

Virtual knockout:
NOT STARTED

Next:
Add Step33C reconstruction after this notebook is validated.


# STEP33C — CellOracle Reconstruction Under Frozen Topology

## Objective

Construct CellOracle input object using frozen biological resources.

Inputs:

- Step23 malignant-only raw count matrix
- Step26 frozen malignant state annotation
- Step29 frozen GRN topology

Rules:

- No GRN rediscovery
- No clustering
- No state redefinition
- No TF reselection
- No RiskScore reconstruction

Only the CellOracle-compatible frozen topology is allowed.

Current stage:
Input reconstruction only.

In [ ]:
from hcc_release_paths import project_path as _hcc_path
from pathlib import Path
import json
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad

print("STEP33C PATH INITIALIZATION")

BASE = Path(_hcc_path(''))

STEP23 = BASE / "LIHC_scRNA_Step23"
STEP26 = BASE / "LIHC_scRNA_Step26"
STEP29 = BASE / "LIHC_scRNA_Step29"

OUT = BASE / "LIHC_scRNA_Step33_Virtual_Knockout"

OUT.mkdir(exist_ok=True, parents=True)

print("BASE:", BASE)
print("OUTPUT:", OUT)

print("Path initialization complete")

In [ ]:
print("Searching frozen h5ad files...")

h5ad_files = list(BASE.rglob("*.h5ad"))

for f in h5ad_files:
    if any(x in str(f) for x in ["Step26", "Step29", "Step23"]):
        print(f)
        
print("Total h5ad:", len(h5ad_files))

In [ ]:
print("Searching frozen h5ad files...")

h5ad_files = list(BASE.rglob("*.h5ad"))

for f in h5ad_files:
    if any(x in str(f) for x in ["Step26", "Step29", "Step23"]):
        print(f)
        
print("Total h5ad:", len(h5ad_files))

In [ ]:
from pathlib import Path
import scanpy as sc

SC_PATH = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step23_Malignant_State_Analysis/Data/"
    "GSE149614_STEP23D3_LINEAGE_CLEAN_RAW_COUNTS.h5ad"
)

adata = sc.read_h5ad(SC_PATH)

print(adata)

print("Cells:", adata.n_obs)
print("Genes:", adata.n_vars)

In [ ]:
print("STEP33C biological annotation audit")

required_obs = [
    "patient",
    "final_malignant_state_id"
]

for col in required_obs:
    print(
        col,
        ":",
        col in adata.obs.columns
    )

print()

if "final_malignant_state_id" in adata.obs.columns:
    print(
        adata.obs["final_malignant_state_id"]
        .value_counts()
    )

In [ ]:
audit = {
    "cells": int(adata.n_obs),
    "genes": int(adata.n_vars),
    "obs_columns": list(adata.obs.columns),
    "stage": "Step33C_input_reconstruction",
    "virtual_knockout_started": False
}

with open(
    OUT / "Step33C_input_audit.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:")
print(OUT / "Step33C_input_audit.json")

# Cell 4 — Step33C Raw Count Integrity Audit

## Objective

Verify the integrity of the frozen Step23 lineage-clean raw count matrix.

Checks:

- Cell barcode uniqueness
- Gene uniqueness
- Count non-negativity
- Count integer property
- Missing value detection

No normalization or transformation is performed.

In [ ]:
# Cell 4 — Step33C Raw Count Integrity Audit

import numpy as np
import pandas as pd
import json


print("="*70)
print("Cell 4 — Raw Count Integrity Audit")
print("="*70)


# -----------------------------
# Basic structure
# -----------------------------

audit_result = {
    "n_cells": int(adata.n_obs),
    "n_genes": int(adata.n_vars),
    "unique_cells": int(adata.obs_names.is_unique),
    "unique_genes": int(adata.var_names.is_unique),
}


# -----------------------------
# Matrix checks
# -----------------------------

X = adata.X


if hasattr(X, "toarray"):
    X_check = X[:1000].toarray()
else:
    X_check = X[:1000]


audit_result["negative_values"] = int(
    (X_check < 0).sum()
)


audit_result["missing_values"] = int(
    np.isnan(X_check).sum()
)


# integer check
integer_like = np.all(
    np.isclose(
        X_check,
        np.round(X_check)
    )
)

audit_result["integer_like"] = bool(integer_like)


# -----------------------------
# Print
# -----------------------------

for k,v in audit_result.items():
    print(f"{k}: {v}")


print("="*70)


# -----------------------------
# Save report
# -----------------------------

with open(
    OUT / "Step33C_RAW_COUNT_INTEGRITY_REPORT.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit_result,
        f,
        indent=2,
        ensure_ascii=False
    )


print("Saved:")
print(
    OUT / "Step33C_RAW_COUNT_INTEGRITY_REPORT.json"
)

# Cell 5 — Load Step26 Frozen MS State Annotation

## Objective

Load frozen malignant state annotation from Step26.

Purpose:

Attach MS1-MS8 state labels to the raw-count matrix.

Rules:

- No reclustering
- No state recalculation
- No modification of labels

Only barcode-level alignment is allowed.

In [ ]:
# Cell 5 — Load Step26 Frozen MS State Annotation

from pathlib import Path
import scanpy as sc


STEP26_PATH = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step26_MS3_Biology/Data/"
    "GSE149614_STEP26_FINAL_FROZEN_MS3_BIOLOGY.h5ad"
)


adata_ms3 = sc.read_h5ad(
    STEP26_PATH
)


print("="*70)
print("Cell 5 — Step26 Frozen MS Annotation")
print("="*70)

print(adata_ms3)

print()

print(
    "Cells:",
    adata_ms3.n_obs
)

print(
    "Genes:",
    adata_ms3.n_vars
)


print()

print(
    "Required column exists:",
    "final_malignant_state_id"
    in adata_ms3.obs.columns
)

print("="*70)

# Cell 6 — Step23 / Step26 Barcode Alignment Audit

## Objective

Verify that raw count input and frozen MS annotation refer to identical cells.

Criterion:

Exact barcode overlap.

Expected:

12057 / 12057 cells matched.

In [ ]:
# Cell 6 — Barcode Alignment Audit


raw_cells = set(
    adata.obs_names
)

state_cells = set(
    adata_ms3.obs_names
)


alignment_report = pd.DataFrame(
    {
        "metric":[
            "raw_count_cells",
            "state_annotation_cells",
            "overlap",
            "raw_only",
            "state_only"
        ],
        "value":[
            len(raw_cells),
            len(state_cells),
            len(raw_cells & state_cells),
            len(raw_cells - state_cells),
            len(state_cells - raw_cells)
        ]
    }
)


display(
    alignment_report
)


alignment_report.to_csv(
    OUT / "Step33C_STATE_ALIGNMENT_REPORT.csv",
    index=False
)


print("Saved:")
print(
    OUT / "Step33C_STATE_ALIGNMENT_REPORT.csv"
)

In [ ]:
# Cell 7 — MS State Distribution Summary


state_summary = (
    adata_ms3.obs[
        "final_malignant_state_id"
    ]
    .value_counts()
    .rename_axis("state")
    .reset_index(
        name="cell_number"
    )
)


display(
    state_summary
)


state_summary.to_csv(
    OUT / "Step33C_MS_STATE_DISTRIBUTION.csv",
    index=False
)


print("Saved:")
print(
    OUT / "Step33C_MS_STATE_DISTRIBUTION.csv"
)

In [ ]:
# Cell 8 — Step33C Summary Report


summary = {
    "stage": "Step33C_Frozen_Input_Reconstruction",
    "raw_count_loaded": True,
    "ms_annotation_loaded": True,
    "barcode_alignment_completed": True,
    "cells": int(adata.n_obs),
    "virtual_knockout_started": False,
    "next_step": "Step33C3_Frozen_GRN_Loading"
}


with open(
    OUT / "Step33C_FINAL_SUMMARY_REPORT.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )


print(json.dumps(
    summary,
    indent=2,
    ensure_ascii=False
))


print()

print("Saved:")
print(
    OUT / "Step33C_FINAL_SUMMARY_REPORT.json"
)

In [ ]:
print()
print("========== STEP33C SUMMARY ==========")

print("Input:")
print("Raw count:", ...)
print("MS annotation:", ...)

print("Alignment:")
print("Matched cells:", ...)

print("QC:")
print("Negative counts:", ...)
print("Integer-like:", ...)

print("Status:")
print("Virtual knockout started:", False)

print("Next:")
print("Step33C3 Frozen GRN Loading")

print("====================================")

# Cell 9 — Step33C3 Frozen Step29 GRN Discovery

## Objective

Locate and verify the frozen Step29 GRN topology.

Rules:

- Use frozen Step29 network only.
- No GRN reconstruction.
- No additional edge inference.

Required:

- TF-target edge table
- Frozen topology checksum
- Compatibility with CellOracle input format

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# Cell 9 — Step33C3 GRN File Discovery


from pathlib import Path


STEP29_DIR = Path(
    _hcc_path('LIHC_scRNA_Step29_GRN')
)


print("="*70)
print("Cell 9 — Searching Step29 GRN files")
print("="*70)


patterns = [
    "*.csv",
    "*.tsv",
    "*.pkl",
    "*.pickle",
    "*.json",
    "*.txt"
]


grn_candidates = []


for pattern in patterns:
    for f in STEP29_DIR.rglob(pattern):
        name = f.name.lower()

        if any(
            key in name
            for key in [
                "grn",
                "network",
                "edge",
                "regulon",
                "tf",
                "target"
            ]
        ):
            grn_candidates.append(f)


print(
    "Candidate files:",
    len(grn_candidates)
)


for i, f in enumerate(grn_candidates):
    print(i, f)


print("="*70)

In [ ]:
# Cell 10 — Step33C3 Load Frozen Regulon Target Network


from pathlib import Path
import json
import pandas as pd


GRN_PATH = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step29_GRN/Data/"
    "Step29F_Regulon_Targets.json"
)


with open(
    GRN_PATH,
    "r",
    encoding="utf-8"
) as f:
    regulon_targets = json.load(f)


print("="*70)
print("Cell 10 — Frozen Step29 Regulon Network Loaded")
print("="*70)


print(
    "Number of regulons:",
    len(regulon_targets)
)


print()

print(
    "Example TF:"
)

example_tf = list(regulon_targets.keys())[0]

print(example_tf)

print(
    regulon_targets[example_tf][:10]
)


print("="*70)

In [ ]:
# Cell 11 — Convert regulon dictionary


edges = []


for tf, targets in regulon_targets.items():

    for target in targets:

        edges.append(
            {
                "TF": tf,
                "target": target
            }
        )


grn_df = pd.DataFrame(edges)


print("="*70)
print("Cell 11 — Frozen GRN Edge Table")
print("="*70)


print(grn_df.head())

print()

print(
    "Total edges:",
    len(grn_df)
)

print(
    "Unique TF:",
    grn_df["TF"].nunique()
)

print(
    "Unique targets:",
    grn_df["target"].nunique()
)


print("="*70)

In [ ]:
# Cell 12 — Frozen GRN Topology Audit

import pandas as pd
import json


print("="*70)
print("Cell 12 — Frozen GRN Topology Audit")
print("="*70)


# --------------------------------------------------
# Original topology
# --------------------------------------------------

grn_original = grn_df.copy()


grn_original["TF_clean"] = (
    grn_original["TF"]
    .str.replace(r"\(\+\)$", "", regex=True)
    .str.replace(r"\(-\)$", "", regex=True)
)


# self-loop audit
self_loop_mask = (
    grn_original["TF_clean"]
    ==
    grn_original["target"]
)


n_self_loops = int(
    self_loop_mask.sum()
)


n_original_edges = int(
    len(grn_original)
)


print("Original edges:", n_original_edges)

print(
    "Self-loops:",
    n_self_loops
)


# --------------------------------------------------
# CellOracle compatible topology
# --------------------------------------------------

grn_simulation = (
    grn_original
    .loc[~self_loop_mask]
    .copy()
)


n_sim_edges = int(
    len(grn_simulation)
)


print(
    "Simulation edges:",
    n_sim_edges
)


print("="*70)



# --------------------------------------------------
# Candidate TF audit
# --------------------------------------------------

candidate_tfs = [
    "FOXM1",
    "MYBL2",
    "E2F8",
    "TFDP1",
    "E2F1"
]


tf_audit = []


for tf in candidate_tfs:

    sub = grn_original[
        grn_original["TF_clean"] == tf
    ]

    tf_audit.append(
        {
            "TF": tf,
            "target_number": int(len(sub)),
            "self_loops": int(
                (
                    sub["TF_clean"]
                    ==
                    sub["target"]
                ).sum()
            )
        }
    )


tf_audit_df = pd.DataFrame(tf_audit)


display(tf_audit_df)


# --------------------------------------------------
# Save
# --------------------------------------------------

grn_simulation.to_csv(
    OUT / "Step33C3_CellOracle_Simulation_Topology.csv",
    index=False
)


tf_audit_df.to_csv(
    OUT / "Step33C3_Target_TF_Audit.csv",
    index=False
)


topology_report = {
    "original_edges": n_original_edges,
    "self_loops_removed": n_self_loops,
    "simulation_edges": n_sim_edges,
    "candidate_TFs": candidate_tfs,
    "next_step": "Step33D_CellOracle_Reconstruction"
}


with open(
    OUT / "Step33C3_TOPOLOGY_AUDIT_REPORT.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        topology_report,
        f,
        indent=2,
        ensure_ascii=False
    )


print()

print("Saved:")
print(
    OUT / "Step33C3_CellOracle_Simulation_Topology.csv"
)

print(
    OUT / "Step33C3_TOPOLOGY_AUDIT_REPORT.json"
)

In [ ]:
# Cell 13 — Step33C3 Final Summary Report


summary = {
    "stage": "Step33C3_Frozen_GRN_Loading",
    "regulon_number": int(grn_df["TF"].nunique()),
    "original_edges": int(len(grn_original)),
    "self_loops_removed": int(n_self_loops),
    "simulation_edges": int(len(grn_simulation)),
    "network_reconstruction": False,
    "TF_reselection": False,
    "next_step": "Step33D_CellOracle_Reconstruction"
}


with open(
    OUT / "Step33C3_FINAL_SUMMARY_REPORT.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )


print("="*70)
print("STEP33C3 Frozen GRN Loading COMPLETE")
print("="*70)

print()

print("Regulons:")
print(summary["regulon_number"])

print()

print("Original edges:")
print(summary["original_edges"])

print("Self-loops removed:")
print(summary["self_loops_removed"])

print("Simulation edges:")
print(summary["simulation_edges"])

print()

print("Network reconstruction:")
print(summary["network_reconstruction"])

print("TF reselection:")
print(summary["TF_reselection"])

print()

print("Next:")
print(summary["next_step"])

print("="*70)


print()

print("Saved:")
print(
    OUT / "Step33C3_FINAL_SUMMARY_REPORT.json"
)

# Cell 14 — Step33D CellOracle Input Preparation

Prepare frozen inputs for CellOracle reconstruction.

Inputs:
- Step23 frozen raw counts
- Step26 frozen MS annotation
- Step33C3 CellOracle simulation topology

No perturbation performed.

In [ ]:
# Cell 14 — Step33D CellOracle Input Preparation

import pandas as pd
import numpy as np
import json
from pathlib import Path


print("="*70)
print("Cell 14 — Step33D CellOracle Input Preparation")
print("="*70)


# check objects exist

required_objects = [
    "adata",
    "adata_ms3",
    "grn_simulation"
]


for obj in required_objects:
    print(
        obj,
        ":",
        obj in globals()
    )


# align cells

common_cells = (
    adata.obs_names
    .intersection(
        adata_ms3.obs_names
    )
)


print()

print(
    "Common cells:",
    len(common_cells)
)


# subset annotation

adata = adata[
    common_cells
].copy()


adata_ms3 = adata_ms3[
    common_cells
].copy()


print()

print(
    "Expression:",
    adata.shape
)

print(
    "Annotation:",
    adata_ms3.shape
)


print("="*70)

In [ ]:
# Cell 15 — CellOracle GRN formatting

import re


grn_celloracle = grn_simulation.copy()


grn_celloracle["TF"] = (
    grn_celloracle["TF"]
    .str.replace(
        r"\(\+\)$",
        "",
        regex=True
    )
    .str.replace(
        r"\(-\)$",
        "",
        regex=True
    )
)


grn_celloracle = (
    grn_celloracle
    .drop_duplicates()
    .reset_index(drop=True)
)


print("="*70)

print(
    "CellOracle GRN edges:",
    len(grn_celloracle)
)

print(
    "TF number:",
    grn_celloracle["TF"].nunique()
)

print(
    "Target number:",
    grn_celloracle["target"].nunique()
)

print("="*70)


grn_celloracle.to_csv(
    OUT / "Step33D_CellOracle_GRN_Input.csv",
    index=False
)


print(
    "Saved:",
    OUT / "Step33D_CellOracle_GRN_Input.csv"
)

In [ ]:
# Cell 16A — Attach frozen Step26 embedding

print("="*70)
print("Attaching frozen Harmony embedding")
print("="*70)


embedding_key = "X_pca_harmony_lineage_clean"


print(
    "Embedding exists in Step26:",
    embedding_key in adata_ms3.obsm
)


if embedding_key not in adata_ms3.obsm:
    raise KeyError(
        f"{embedding_key} not found in Step26 object"
    )


# align order

adata.obsm[embedding_key] = (
    adata_ms3
    .obsm[embedding_key]
    .copy()
)


print(
    "Embedding shape:",
    adata.obsm[embedding_key].shape
)


print("="*70)

In [ ]:
# Cell 16B — Attach frozen MS state labels

print("="*70)
print("Cell 16B — Attaching frozen MS state annotation")
print("="*70)


state_key = "final_malignant_state_id"


print(
    "State exists in Step26:",
    state_key in adata_ms3.obs.columns
)


if state_key not in adata_ms3.obs.columns:
    raise KeyError(
        f"{state_key} not found in Step26 annotation"
    )


# align by barcode

adata.obs[state_key] = (
    adata_ms3
    .obs.loc[
        adata.obs_names,
        state_key
    ]
    .copy()
)


print()

print(
    "State labels added:",
    state_key in adata.obs.columns
)


print(
    adata.obs[state_key]
    .value_counts()
)


print("="*70)

In [ ]:
# Cell 16 — Build CellOracle Oracle Object

import celloracle as co


print("="*70)
print("Cell 16 — Creating CellOracle Oracle object")
print("="*70)


# --------------------------------------------------
# Create Oracle object
# --------------------------------------------------

oracle = co.Oracle()


# --------------------------------------------------
# Import frozen raw counts
# --------------------------------------------------

oracle.import_anndata_as_raw_count(
    adata=adata,
    cluster_column_name="final_malignant_state_id",
    embedding_name="X_pca_harmony_lineage_clean"
)


print()
print("Oracle object created successfully")


print()

print("Cells:")
print(oracle.adata.n_obs)

print("Genes:")
print(oracle.adata.n_vars)


print()

print("Embedding:")
print(
    oracle.embedding.shape
)


print("="*70)

In [ ]:
# Cell 17 — Import Frozen GRN as TFdict


print("="*70)
print("Cell 17 — Importing frozen GRN as TFdict")
print("="*70)


# Build TF dictionary

TFdict = (
    grn_celloracle
    .groupby("TF")["target"]
    .apply(list)
    .to_dict()
)


print(
    "Number of TF:",
    len(TFdict)
)


print()

example_tf = list(TFdict.keys())[0]

print(
    "Example TF:",
    example_tf
)

print(
    "Targets:",
    TFdict[example_tf][:10]
)


# Import into Oracle

oracle.import_TF_data(
    TFdict=TFdict
)


print()

print("Frozen GRN imported successfully")

print("="*70)

In [ ]:
# Cell 18 — GRN gene matching audit


print("="*70)
print("Cell 18 — GRN Gene Matching Audit")
print("="*70)


adata_genes = set(
    oracle.adata.var_names
)


grn_tfs = set(
    grn_celloracle["TF"]
)


grn_targets = set(
    grn_celloracle["target"]
)


matched_tf = (
    adata_genes
    &
    grn_tfs
)


matched_target = (
    adata_genes
    &
    grn_targets
)


print(
    "Expression genes:",
    len(adata_genes)
)

print(
    "GRN TF:",
    len(grn_tfs)
)

print(
    "Matched TF:",
    len(matched_tf)
)

print()

print(
    "GRN targets:",
    len(grn_targets)
)

print(
    "Matched targets:",
    len(matched_target)
)


print("="*70)

In [ ]:
# Cell 19 — Candidate TF coverage


candidate_TFs = [
    "FOXM1",
    "MYBL2",
    "E2F8",
    "TFDP1",
    "E2F1"
]


candidate_report = []


for tf in candidate_TFs:

    edges = grn_celloracle[
        grn_celloracle["TF"] == tf
    ]

    candidate_report.append(
        {
            "TF": tf,
            "target_number": len(edges),
            "TF_present_in_expression":
                tf in oracle.adata.var_names,
            "target_present_number":
                sum(
                    edges["target"]
                    .isin(
                        oracle.adata.var_names
                    )
                )
        }
    )


candidate_report = pd.DataFrame(
    candidate_report
)


display(
    candidate_report
)


candidate_report.to_csv(
    OUT / "Step33D_Candidate_TF_Coverage.csv",
    index=False
)


print("Saved:")
print(
    OUT / "Step33D_Candidate_TF_Coverage.csv"
)

In [ ]:
# Cell 20 — Gene filtering preparation


print("="*70)
print("Cell 20 — CellOracle gene filtering preparation")
print("="*70)


# genes from frozen GRN

grn_genes = set(
    grn_celloracle["TF"]
).union(
    set(grn_celloracle["target"])
)


adata_genes = set(
    oracle.adata.var_names
)


overlap_genes = (
    grn_genes
    &
    adata_genes
)


print(
    "Frozen GRN genes:",
    len(grn_genes)
)

print(
    "Expression genes:",
    len(adata_genes)
)

print(
    "Overlap genes:",
    len(overlap_genes)
)


print("="*70)

In [ ]:
# Cell 21 — Filter AnnData for CellOracle


fit_genes = sorted(
    list(overlap_genes)
)


adata_fit = adata[
    :,
    fit_genes
].copy()


print("="*70)

print(
    "Filtered AnnData:"
)

print(
    adata_fit
)

print()

print(
    "Cells:",
    adata_fit.n_obs
)

print(
    "Genes:",
    adata_fit.n_vars
)

print("="*70)

In [ ]:
# Cell 22 — Build fitting Oracle


oracle_fit = co.Oracle()


oracle_fit.import_anndata_as_raw_count(
    adata=adata_fit,
    cluster_column_name="final_malignant_state_id",
    embedding_name="X_pca_harmony_lineage_clean"
)


oracle_fit.import_TF_data(
    TFdict=TFdict
)


print("="*70)

print("Fitting Oracle created")

print(
    oracle_fit.adata.shape
)

print("="*70)

In [ ]:
# Cell 23A — CellOracle PCA and KNN imputation


print("="*70)
print("Cell 23A — CellOracle PCA and KNN imputation")
print("="*70)


# check function signature

import inspect


print(
    inspect.signature(
        oracle_fit.perform_PCA
    )
)


print()

print(
    inspect.signature(
        oracle_fit.knn_imputation
    )
)


print("="*70)

In [ ]:
# Cell 23A — CellOracle PCA and KNN imputation


print("="*70)
print("Cell 23A — CellOracle PCA and KNN imputation")
print("="*70)


# PCA
oracle_fit.perform_PCA(
    n_components=30
)


print("PCA completed")


# KNN imputation
oracle_fit.knn_imputation(
    k=10,
    n_pca_dims=30,
    n_jobs=8
)


print()

print("KNN imputation completed")


print()

print(
    "Available layers:"
)

print(
    list(
        oracle_fit.adata.layers.keys()
    )
)


print("="*70)

In [ ]:
# Cell 23 — Fit GRN coefficients


print("="*70)
print("Cell 23 — CellOracle GRN coefficient fitting")
print("="*70)


oracle_fit.fit_GRN_for_simulation(
    alpha=10
)


print()

print("GRN fitting completed")

print("="*70)

In [ ]:
# Cell 24 — GRN fitting QC audit


print("="*70)
print("Cell 24 — Step33D GRN fitting QC")
print("="*70)


# check fitted result

print(
    "Oracle attributes:"
)

for attr in [
    "coef_matrix",
    "GRN_unit",
    "TFdict",
    "links"
]:
    print(
        attr,
        ":",
        hasattr(oracle_fit, attr)
    )


print()


# cluster information

if hasattr(
    oracle_fit,
    "cluster_column_name"
):
    print(
        "Cluster column:",
        oracle_fit.cluster_column_name
    )


print()

print(
    "Cells:",
    oracle_fit.adata.n_obs
)

print(
    "Genes:",
    oracle_fit.adata.n_vars
)


print("="*70)

In [ ]:
# Cell 25 — FOXM1 coefficient audit


print("="*70)
print("Cell 25 — FOXM1 coefficient audit")
print("="*70)


foxm1_targets = (
    grn_celloracle[
        grn_celloracle["TF"]=="FOXM1"
    ]["target"]
    .tolist()
)


print(
    "FOXM1 frozen targets:",
    len(foxm1_targets)
)


available_targets = [
    g for g in foxm1_targets
    if g in oracle_fit.adata.var_names
]


print(
    "Targets in fitting matrix:",
    len(available_targets)
)


print()

print(
    "Example targets:"
)

print(
    available_targets[:20]
)


print("="*70)

In [ ]:
# Cell 26 — FOXM1 Virtual Knockout Pre-simulation Check


print("="*70)
print("Cell 26 — FOXM1 Virtual Knockout Pre-simulation Check")
print("="*70)


KO_TF = "FOXM1"


# check TF existence

tf_exists = KO_TF in oracle_fit.TFdict


print(
    "FOXM1 in TFdict:",
    tf_exists
)


# frozen targets

FOXM1_targets = (
    grn_celloracle[
        grn_celloracle["TF"] == KO_TF
    ]["target"]
    .drop_duplicates()
    .tolist()
)


print(
    "Frozen FOXM1 targets:",
    len(FOXM1_targets)
)


# remove self if exists

FOXM1_targets_no_self = [
    x for x in FOXM1_targets
    if x != KO_TF
]


print(
    "Targets after self-loop removal:",
    len(FOXM1_targets_no_self)
)


print()

print(
    "Target preview:"
)

print(
    FOXM1_targets_no_self[:30]
)


# save lock

foxm1_lock = {
    "TF": KO_TF,
    "original_targets": len(FOXM1_targets),
    "simulation_targets": len(FOXM1_targets_no_self),
    "self_removed": len(FOXM1_targets)-len(FOXM1_targets_no_self),
    "simulation_started": False
}


with open(
    OUT / "Step33D_FOXM1_KO_Precheck.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        foxm1_lock,
        f,
        indent=2,
        ensure_ascii=False
    )


print()

print("Saved:")
print(
    OUT / "Step33D_FOXM1_KO_Precheck.json"
)


print("="*70)

In [ ]:
# Cell 27A — FOXM1 Simulation Eligibility Audit


print("="*70)
print("Cell 27A — FOXM1 Simulation Eligibility Audit")
print("="*70)


KO_TF = "FOXM1"


print(
    "FOXM1 in oracle_fit.TFdict:",
    KO_TF in oracle_fit.TFdict
)


print(
    "FOXM1 in all_regulatory_genes_in_TFdict:",
    KO_TF in oracle_fit.all_regulatory_genes_in_TFdict
)


print()

print(
    "Number of regulatory genes:",
    len(oracle_fit.all_regulatory_genes_in_TFdict)
)


print()

print(
    "FOXM1 related keys:"
)

for x in oracle_fit.all_regulatory_genes_in_TFdict:
    if "FOXM1" in x.upper():
        print(x)


print("="*70)

In [ ]:
# Cell 27B — Register FOXM1 regulatory node (list version)


print("="*70)
print("Cell 27B — Register FOXM1 regulatory node")
print("="*70)


KO_TF = "FOXM1"


print(
    "Object type:",
    type(
        oracle_fit.all_regulatory_genes_in_TFdict
    )
)


print(
    "Before:",
    KO_TF in oracle_fit.all_regulatory_genes_in_TFdict
)


if KO_TF not in oracle_fit.all_regulatory_genes_in_TFdict:
    
    oracle_fit.all_regulatory_genes_in_TFdict.append(
        KO_TF
    )


print(
    "After:",
    KO_TF in oracle_fit.all_regulatory_genes_in_TFdict
)


print(
    "Regulatory genes:",
    len(
        oracle_fit.all_regulatory_genes_in_TFdict
    )
)


print("="*70)

In [ ]:
# Cell 28A — FOXM1 active regulatory audit


print("="*70)
print("Cell 28A — FOXM1 active regulatory audit")
print("="*70)


KO_TF = "FOXM1"


print(
    "TFdict:",
    KO_TF in oracle_fit.TFdict
)


print(
    "all_regulatory:",
    KO_TF in oracle_fit.all_regulatory_genes_in_TFdict
)


print(
    "active_regulatory:",
    KO_TF in oracle_fit.active_regulatory_genes
)


print()

print(
    "Active regulatory genes:",
    len(
        oracle_fit.active_regulatory_genes
    )
)


print()

print(
    "FOXM1 coefficient related targets:"
)


if hasattr(
    oracle_fit,
    "links"
):
    print(
        "links exists"
    )


print("="*70)

In [ ]:
# Cell 28C — Inspect fitted objects


print("="*70)
print("Cell 28C — Inspect CellOracle fitted objects")
print("="*70)


attrs = [
    "links",
    "coef_matrix",
    "coef",
    "GRN",
    "active_regulatory_genes",
    "high_var_genes",
    "all_regulatory_genes_in_TFdict"
]


for a in attrs:
    print(
        a,
        ":",
        hasattr(
            oracle_fit,
            a
        )
    )


print()

print("Non-private attributes containing coef/link/GRN:")


for x in dir(oracle_fit):
    if (
        "coef" in x.lower()
        or
        "link" in x.lower()
        or
        "grn" in x.lower()
    ):
        print(x)


print("="*70)

In [ ]:
# Cell 28D — Correct CellOracle TFdict orientation

print("=" * 70)
print("Cell 28D — Correcting CellOracle TFdict orientation")
print("=" * 70)

# CellOracle expects:
# target_gene -> list of candidate regulatory TFs

TFdict_correct = (
    grn_celloracle
    .groupby("target")["TF"]
    .apply(lambda x: sorted(set(x)))
    .to_dict()
)

print("Target genes in TFdict:", len(TFdict_correct))

print("Example target:")
example_target = next(iter(TFdict_correct))
print(example_target, "->", TFdict_correct[example_target][:10])

# FOXM1 should appear as a regulator of its frozen downstream targets
foxm1_regulated_targets = [
    target
    for target, regulators in TFdict_correct.items()
    if "FOXM1" in regulators
]

print()
print("FOXM1 regulated targets:", len(foxm1_regulated_targets))
print("Preview:", foxm1_regulated_targets[:20])

assert len(foxm1_regulated_targets) == 58, (
    f"Expected 58 FOXM1 targets, found {len(foxm1_regulated_targets)}"
)

print("=" * 70)
print("TFdict orientation audit: PASS")

In [ ]:
# Cell 29 — Rebuild fitting Oracle with corrected TFdict

import celloracle as co

print("=" * 70)
print("Cell 29 — Rebuilding CellOracle object")
print("=" * 70)

oracle_fit_v2 = co.Oracle()

oracle_fit_v2.import_anndata_as_raw_count(
    adata=adata_fit,
    cluster_column_name="final_malignant_state_id",
    embedding_name="X_pca_harmony_lineage_clean"
)

oracle_fit_v2.import_TF_data(
    TFdict=TFdict_correct
)

print("Oracle rebuilt successfully")
print("Cells:", oracle_fit_v2.adata.n_obs)
print("Genes:", oracle_fit_v2.adata.n_vars)

print()
print("FOXM1 in all_regulatory_genes_in_TFdict:",
      "FOXM1" in oracle_fit_v2.all_regulatory_genes_in_TFdict)

print("=" * 70)

In [ ]:
# Cell 29 — Simulation output audit


print("="*70)
print("Cell 29 — Simulation output audit")
print("="*70)


attrs = [
    "delta_X",
    "shift",
    "simulation",
    "simulation_input"
]


for a in attrs:
    print(
        a,
        ":",
        hasattr(
            oracle_fit,
            a
        )
    )


print()


print(
    "Available layers:"
)

print(
    list(
        oracle_fit.adata.layers.keys()
    )
)


print("="*70)

In [ ]:

import os
import sys

print("Python:", sys.executable)
print("Working directory:", os.getcwd())

In [ ]:
import psutil

mem = psutil.virtual_memory()

print("Total RAM GB:", round(mem.total / 1024**3, 2))
print("Available RAM GB:", round(mem.available / 1024**3, 2))
print("Used %:", mem.percent)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# Cell 33A — Rebuild frozen simulation topology and correct TFdict

from pathlib import Path
import json
import re
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import celloracle as co

OUT = Path(_hcc_path('LIHC_scRNA_Step33_Virtual_Knockout'))
OUT.mkdir(parents=True, exist_ok=True)

RAW_PATH = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step23_Malignant_State_Analysis/Data/"
    "GSE149614_STEP23D3_LINEAGE_CLEAN_RAW_COUNTS.h5ad"
)

ANN_PATH = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step26_MS3_Biology/Data/"
    "GSE149614_STEP26_FINAL_FROZEN_MS3_BIOLOGY.h5ad"
)

REG_PATH = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step29_GRN/Data/"
    "Step29F_Regulon_Targets.json"
)

print("=" * 70)
print("Cell 33A — Frozen topology + TFdict recovery")
print("=" * 70)

with open(REG_PATH, "r", encoding="utf-8") as f:
    regulons = json.load(f)

rows = []

for tf_raw, targets in regulons.items():
    tf = re.sub(r"\([+-]\)$", "", tf_raw)

    for target in targets:
        rows.append((tf, target))

grn_original = pd.DataFrame(
    rows,
    columns=["TF", "target"]
).drop_duplicates()

self_mask = grn_original["TF"] == grn_original["target"]

grn_sim = (
    grn_original.loc[~self_mask]
    .copy()
    .reset_index(drop=True)
)

assert len(grn_original) == 32530
assert int(self_mask.sum()) == 140
assert len(grn_sim) == 32390

# CellOracle-required orientation:
# target -> regulatory TF list
TFdict_final = (
    grn_sim
    .groupby("target")["TF"]
    .apply(lambda x: sorted(set(x)))
    .to_dict()
)

foxm1_targets = [
    target
    for target, regulators in TFdict_final.items()
    if "FOXM1" in regulators
]

assert len(foxm1_targets) == 58

print("Original topology :", len(grn_original))
print("Self-loops removed:", int(self_mask.sum()))
print("Simulation edges  :", len(grn_sim))
print("TFdict targets    :", len(TFdict_final))
print("FOXM1 targets     :", len(foxm1_targets))
print("=" * 70)

In [ ]:
# Cell 33B — Rebuild fitting AnnData

print("=" * 70)
print("Cell 33B — Rebuild fitting AnnData")
print("=" * 70)

adata_raw = sc.read_h5ad(RAW_PATH)
adata_ann = sc.read_h5ad(ANN_PATH)

assert adata_raw.n_obs == 12057
assert adata_ann.n_obs == 12057

# Exact barcode alignment
idx = adata_ann.obs_names.get_indexer(adata_raw.obs_names)

assert np.all(idx >= 0)
assert len(np.unique(idx)) == adata_raw.n_obs

state_key = "final_malignant_state_id"
embed_key = "X_pca_harmony_lineage_clean"

adata_raw.obs[state_key] = (
    adata_ann.obs.iloc[idx][state_key]
    .astype(str)
    .values
)

adata_raw.obs[state_key] = (
    adata_raw.obs[state_key]
    .astype("category")
)

adata_raw.obsm[embed_key] = (
    np.asarray(
        adata_ann.obsm[embed_key]
    )[idx, :]
    .copy()
)

# Frozen simulation gene universe
fit_genes = sorted(
    set(grn_sim["TF"])
    |
    set(grn_sim["target"])
)

missing_genes = sorted(
    set(fit_genes)
    -
    set(adata_raw.var_names)
)

assert len(missing_genes) == 0

adata_fit = adata_raw[:, fit_genes].copy()

print("adata_fit:", adata_fit.shape)
print("States:")
print(adata_fit.obs[state_key].value_counts().sort_index())

assert adata_fit.shape == (12057, 10776)

# Free source objects BEFORE CellOracle expansion
del adata_raw
del adata_ann
gc.collect()

print()
print("Source AnnData objects released from memory.")
print("=" * 70)

In [ ]:
# Cell 33C — Build formal corrected CellOracle object

print("=" * 70)
print("Cell 33C — Build formal CellOracle object")
print("=" * 70)

oracle_final = co.Oracle()

oracle_final.import_anndata_as_raw_count(
    adata=adata_fit,
    cluster_column_name="final_malignant_state_id",
    embedding_name="X_pca_harmony_lineage_clean"
)

oracle_final.import_TF_data(
    TFdict=TFdict_final
)

print(
    "FOXM1 regulatory TF:",
    "FOXM1" in oracle_final.all_regulatory_genes_in_TFdict
)

print(
    "Cells / genes:",
    oracle_final.adata.shape
)

assert "FOXM1" in oracle_final.all_regulatory_genes_in_TFdict

# Oracle already owns its AnnData; avoid holding a second big copy
del adata_fit
gc.collect()

print("=" * 70)
print("Corrected Oracle import: PASS")

In [ ]:
# Cell 33D — Frozen CellOracle PCA/KNN preprocessing

import psutil

print("=" * 70)
print("Cell 33D — Formal PCA + KNN imputation")
print("=" * 70)

# PCA
oracle_final.perform_PCA()

evr = oracle_final.pca.explained_variance_ratio_

pc_candidates = np.where(
    np.diff(
        np.diff(
            np.cumsum(evr)
        ) > 0.002
    )
)[0]

if len(pc_candidates) == 0:
    n_comps = min(50, len(evr))
else:
    n_comps = min(
        int(pc_candidates[0]),
        50
    )

n_comps = max(n_comps, 2)

# Frozen CellOracle KNN rule
n_cells = oracle_final.adata.n_obs
k = int(0.025 * n_cells)

b_sight = k * 8
b_maxl = k * 4

print("Cells       :", n_cells)
print("Selected PCs:", n_comps)
print("k           :", k)
print("b_sight     :", b_sight)
print("b_maxl      :", b_maxl)

assert n_cells == 12057
assert k == 301

gc.collect()

mem = psutil.virtual_memory()
print(
    "Available RAM before KNN:",
    round(mem.available / 1024**3, 2),
    "GB"
)

oracle_final.knn_imputation(
    n_pca_dims=n_comps,
    k=k,
    balanced=True,
    b_sight=b_sight,
    b_maxl=b_maxl,
    n_jobs=4
)

print()
print("Layers:")
print(list(oracle_final.adata.layers.keys()))

assert "imputed_count" in oracle_final.adata.layers

gc.collect()

mem = psutil.virtual_memory()
print(
    "Available RAM after KNN:",
    round(mem.available / 1024**3, 2),
    "GB"
)

print("=" * 70)
print("Formal PCA + KNN: PASS")

In [ ]:
# Cell 33E — Formal GRN fitting + candidate TF eligibility audit

import gc
import psutil
import pandas as pd

print("=" * 70)
print("Cell 33E — Formal GRN coefficient fitting")
print("=" * 70)

mem = psutil.virtual_memory()
print(
    "Available RAM before fitting:",
    round(mem.available / 1024**3, 2),
    "GB"
)

# --------------------------------------------------
# Formal fitting
# --------------------------------------------------

oracle_final.fit_GRN_for_simulation(
    alpha=10
)

gc.collect()

print()
print("GRN coefficient fitting completed.")

# --------------------------------------------------
# Audit all formal Step33 TF targets
# --------------------------------------------------

FORMAL_TFS = [
    "FOXM1",
    "MYBL2",
    "E2F8",
    "TFDP1",
    "E2F1"
]

rows = []

for tf in FORMAL_TFS:

    frozen_targets = (
        grn_sim.loc[
            grn_sim["TF"] == tf,
            "target"
        ]
        .drop_duplicates()
        .tolist()
    )

    rows.append({
        "TF": tf,
        "frozen_targets": len(frozen_targets),
        "in_all_regulatory_genes":
            tf in oracle_final.all_regulatory_genes_in_TFdict,
        "in_active_regulatory_genes":
            tf in oracle_final.active_regulatory_genes,
        "in_high_var_genes":
            tf in oracle_final.high_var_genes,
    })

eligibility_df = pd.DataFrame(rows)

print()
print(eligibility_df.to_string(index=False))

# Save
ELIG_PATH = OUT / "Step33_FORMAL_TF_ELIGIBILITY.csv"

eligibility_df.to_csv(
    ELIG_PATH,
    index=False
)

print()
print("Saved:")
print(ELIG_PATH)

mem = psutil.virtual_memory()

print()
print(
    "Available RAM after fitting:",
    round(mem.available / 1024**3, 2),
    "GB"
)

print("=" * 70)

In [ ]:
# Cell 33F — Save fitted Oracle + formal recovery report

import json
import gc

print("=" * 70)
print("Cell 33F — Save fitted Oracle + recovery report")
print("=" * 70)

# --------------------------------------------------
# Save fitted CellOracle object
# --------------------------------------------------

ORACLE_SAVE = (
    OUT /
    "Step33_CellOracle_FORMAL_FITTED.celloracle.oracle"
)

oracle_final.to_hdf5(
    str(ORACLE_SAVE)
)

print("Fitted Oracle saved:")
print(ORACLE_SAVE)

# --------------------------------------------------
# Formal report
# --------------------------------------------------

report = {
    "status": "FORMAL_CELLORACLE_FITTING_COMPLETE",

    "cells": int(oracle_final.adata.n_obs),
    "genes": int(oracle_final.adata.n_vars),

    "original_edges": int(len(grn_original)),
    "removed_self_loops": int(
        (grn_original["TF"] == grn_original["target"]).sum()
    ),
    "simulation_edges": int(len(grn_sim)),

    "TFdict_orientation":
        "target_gene -> regulatory_TFs",

    "n_pca_dims": int(n_comps),

    "knn_k": int(k),
    "balanced_knn": True,
    "b_sight": int(b_sight),
    "b_maxl": int(b_maxl),

    "ridge_alpha": 10,

    "formal_KO_TFs": FORMAL_TFS,

    "network_reconstruction": False,
    "TF_reselection": False,

    "old_reversed_TFdict_attempt_used": False,
    "old_k10_attempt_used": False
}

REPORT_PATH = (
    OUT /
    "Step33_FORMAL_RECOVERY_REPORT.json"
)

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )

gc.collect()

print()
print("Recovery report saved:")
print(REPORT_PATH)

print()
print("=" * 70)
print("FORMAL CELLORACLE RECOVERY COMPLETE")
print("=" * 70)

# Cell 34–37 — FOXM1 Pilot Virtual Knockout

Formal Step33 pilot perturbation:

- TF: FOXM1
- n_propagation = 3
- use_randomized_GRN = False
- clip_delta_X = False
- Frozen Step29 topology
- No GRN reconstruction
- No TF reselection

This module performs simulation and technical QC only.
Biological endpoint interpretation is deferred until simulation QC passes.

In [ ]:
# Cell 34 — FOXM1 simulation API and eligibility check

import inspect
import gc
import psutil

print("=" * 70)
print("Cell 34 — FOXM1 simulation API + eligibility check")
print("=" * 70)

KO_TF = "FOXM1"

print("simulate_shift signature:")
print(inspect.signature(oracle_final.simulate_shift))

print()
print("evaluate_simulated_gene_distribution_range:")
print(inspect.signature(
    oracle_final.evaluate_simulated_gene_distribution_range
))

print()

if hasattr(
    oracle_final,
    "evaluate_and_plot_simulation_value_distribution"
):
    print("evaluate_and_plot_simulation_value_distribution:")
    print(inspect.signature(
        oracle_final.evaluate_and_plot_simulation_value_distribution
    ))

print()
print("FOXM1 checks:")
print(
    "all_regulatory :",
    KO_TF in oracle_final.all_regulatory_genes_in_TFdict
)
print(
    "active         :",
    KO_TF in oracle_final.active_regulatory_genes
)
print(
    "high_var       :",
    KO_TF in oracle_final.high_var_genes
)

assert KO_TF in oracle_final.all_regulatory_genes_in_TFdict
assert KO_TF in oracle_final.active_regulatory_genes
assert KO_TF in oracle_final.high_var_genes

gc.collect()

mem = psutil.virtual_memory()

print()
print(
    "Available RAM:",
    round(mem.available / 1024**3, 2),
    "GB"
)

print("=" * 70)
print("FOXM1 simulation eligibility: PASS")

In [ ]:
# Cell 35 — Formal FOXM1 virtual knockout

import gc
import psutil

print("=" * 70)
print("Cell 35 — Formal FOXM1 virtual knockout")
print("=" * 70)

KO_TF = "FOXM1"

mem = psutil.virtual_memory()

print(
    "Available RAM before simulation:",
    round(mem.available / 1024**3, 2),
    "GB"
)

oracle_final.simulate_shift(
    perturb_condition={
        KO_TF: 0.0
    },
    n_propagation=3,
    use_randomized_GRN=False,
    clip_delta_X=False
)

gc.collect()

print()
print("FOXM1 simulation completed.")

print()
print("Available layers:")
print(list(oracle_final.adata.layers.keys()))

mem = psutil.virtual_memory()

print()
print(
    "Available RAM after simulation:",
    round(mem.available / 1024**3, 2),
    "GB"
)

print("=" * 70)

In [ ]:
# Cell 36 — FOXM1 simulation output audit

import numpy as np
import scipy.sparse as sp

print("=" * 70)
print("Cell 36 — FOXM1 simulation output audit")
print("=" * 70)

layers = list(oracle_final.adata.layers.keys())

print("Layers:")
print(layers)

assert "imputed_count" in layers
assert "simulated_count" in layers

gene_idx = oracle_final.adata.var_names.get_loc("FOXM1")


def get_gene_vector(layer_name, idx):
    x = oracle_final.adata.layers[layer_name][:, idx]

    if sp.issparse(x):
        x = x.toarray()

    return np.asarray(x).ravel()


foxm1_baseline = get_gene_vector(
    "imputed_count",
    gene_idx
)

foxm1_simulated = get_gene_vector(
    "simulated_count",
    gene_idx
)

foxm1_delta = (
    foxm1_simulated
    -
    foxm1_baseline
)

print()
print("FOXM1 baseline:")
print(
    "median =",
    float(np.median(foxm1_baseline)),
    "| mean =",
    float(np.mean(foxm1_baseline))
)

print()

print("FOXM1 simulated:")
print(
    "median =",
    float(np.median(foxm1_simulated)),
    "| mean =",
    float(np.mean(foxm1_simulated))
)

print()

print("FOXM1 delta:")
print(
    "median =",
    float(np.median(foxm1_delta)),
    "| mean =",
    float(np.mean(foxm1_delta))
)

print()

print(
    "Simulated FOXM1 min/max:",
    float(np.min(foxm1_simulated)),
    float(np.max(foxm1_simulated))
)

print("=" * 70)

In [ ]:
# Cell 37A — Built-in simulation QC

print("=" * 70)
print("Cell 37A — CellOracle simulation QC")
print("=" * 70)

qc_range_result = (
    oracle_final
    .evaluate_simulated_gene_distribution_range()
)

print()
print("Distribution-range QC completed.")

print("=" * 70)

In [ ]:
# Cell 37B — Simulation value distribution diagnostic

print("=" * 70)
print("Cell 37B — Simulation value distribution diagnostic")
print("=" * 70)

if hasattr(
    oracle_final,
    "evaluate_and_plot_simulation_value_distribution"
):

    sig = inspect.signature(
        oracle_final.evaluate_and_plot_simulation_value_distribution
    )

    print("Signature:")
    print(sig)

    params = sig.parameters

    # Use a limited diagnostic sample if API exposes n_comparison
    if "n_comparison" in params:
        oracle_final.evaluate_and_plot_simulation_value_distribution(
            n_comparison=10
        )
    else:
        oracle_final.evaluate_and_plot_simulation_value_distribution()

else:
    print(
        "Method not available in this CellOracle version."
    )

print("=" * 70)

In [ ]:
# Cell 37C — FOXM1 pilot QC report

import json
from pathlib import Path

QC_REPORT = {
    "status": "FOXM1_PILOT_SIMULATION_COMPLETED",

    "TF": "FOXM1",

    "n_propagation": 3,
    "use_randomized_GRN": False,
    "clip_delta_X": False,

    "baseline_layer": "imputed_count",
    "simulated_layer": "simulated_count",

    "FOXM1_baseline_median":
        float(np.median(foxm1_baseline)),

    "FOXM1_simulated_median":
        float(np.median(foxm1_simulated)),

    "FOXM1_delta_median":
        float(np.median(foxm1_delta)),

    "FOXM1_simulated_min":
        float(np.min(foxm1_simulated)),

    "FOXM1_simulated_max":
        float(np.max(foxm1_simulated)),

    "biological_endpoint_scoring_started": False,

    "network_reconstruction": False,
    "TF_reselection": False
}

QC_PATH = (
    OUT /
    "Step33_FOXM1_PILOT_SIMULATION_QC_REPORT.json"
)

with open(
    QC_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        QC_REPORT,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 70)
print("Cell 37C — FOXM1 PILOT QC REPORT")
print("=" * 70)

for k_, v_ in QC_REPORT.items():
    print(f"{k_}: {v_}")

print()
print("Saved:")
print(QC_PATH)

print("=" * 70)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# Cell 38 — Locate frozen endpoint gene-set sources

from pathlib import Path

print("=" * 70)
print("Cell 38 — Locate frozen endpoint gene-set sources")
print("=" * 70)

ROOT = Path(_hcc_path(''))

keywords = [
    "Core50",
    "G2M_Independent",
    "Mitotic_Entry",
    "Spindle_Chromosome",
    "Hepatic_Differentiation",
    "Bile_Acid"
]

valid_suffix = {
    ".csv", ".tsv", ".txt", ".json"
}

candidate_files = []

for p in ROOT.rglob("*"):
    if not p.is_file():
        continue

    if p.suffix.lower() not in valid_suffix:
        continue

    # only Step25 / Step31-related material
    path_text = str(p).lower()

    if ("step25" not in path_text) and ("step31" not in path_text):
        continue

    filename_hits = [
        k for k in keywords
        if k.lower() in p.name.lower()
    ]

    content_hits = []

    # Only inspect reasonably small text files
    try:
        if p.stat().st_size < 20 * 1024 * 1024:
            txt = p.read_text(
                encoding="utf-8",
                errors="ignore"
            )

            content_hits = [
                k for k in keywords
                if k.lower() in txt.lower()
            ]

    except Exception:
        pass

    hits = sorted(
        set(filename_hits + content_hits)
    )

    if hits:
        candidate_files.append(
            {
                "file": str(p),
                "hits": hits
            }
        )

print("Candidate files found:", len(candidate_files))
print()

for x in candidate_files:
    print("FILE:")
    print(x["file"])
    print("MATCH:", ", ".join(x["hits"]))
    print("-" * 70)

print("=" * 70)

In [ ]:
# Cell 39 — Frozen endpoint schema audit

import pandas as pd
from pathlib import Path

print("=" * 70)
print("Cell 39 — Frozen endpoint schema audit")
print("=" * 70)

CORE50_PATH = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step25_CrossPatient_Robustness/"
    "Reference/Step25_FROZEN_MS3_Conserved_Core50.csv"
)

PROGRAM_MAP_PATH = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step31_Independent_Validation/"
    "Audit/Step31E2A_ValidationProgramMapping.csv"
)

# --------------------------------------------------
# Core50
# --------------------------------------------------

core50_df = pd.read_csv(CORE50_PATH)

print("\n[Core50]")
print("Path:", CORE50_PATH)
print("Shape:", core50_df.shape)
print("Columns:", core50_df.columns.tolist())
print()
print(core50_df.head(10).to_string(index=False))

# --------------------------------------------------
# Step31 frozen program mapping
# --------------------------------------------------

program_map_df = pd.read_csv(PROGRAM_MAP_PATH)

print("\n" + "-" * 70)
print("[Step31 program mapping]")
print("Path:", PROGRAM_MAP_PATH)
print("Shape:", program_map_df.shape)
print("Columns:", program_map_df.columns.tolist())
print()
print(program_map_df.head(20).to_string(index=False))

print("\nUnique values for low-cardinality columns:")

for col in program_map_df.columns:
    nunique = program_map_df[col].nunique(dropna=True)

    if nunique <= 20:
        print(f"\n{col} ({nunique} unique):")
        print(
            program_map_df[col]
            .dropna()
            .astype(str)
            .value_counts()
            .to_string()
        )

print("=" * 70)

# Cell 40–41 — Frozen Endpoint Lock and Coverage Audit

Frozen endpoint sources:

- Core50: Step25 frozen conserved MS3 Core50
- Independent biological programs: Step31 validation mapping
- Only `mapping_status == retained` is used for Step31 independent programs
- FOXM1 primary Core50 endpoint uses leave-target-out: 49 genes

No endpoint scoring is performed until complete coverage is confirmed.

In [ ]:
# Cell 40 — Load and lock frozen Step33 endpoint gene sets

import pandas as pd
import json
from pathlib import Path

print("=" * 70)
print("Cell 40 — Frozen endpoint lock")
print("=" * 70)

CORE50_PATH = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step25_CrossPatient_Robustness/"
    "Reference/Step25_FROZEN_MS3_Conserved_Core50.csv"
)

PROGRAM_MAP_PATH = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step31_Independent_Validation/"
    "Audit/Step31E2A_ValidationProgramMapping.csv"
)

# --------------------------------------------------
# Core50
# --------------------------------------------------

core50_df = pd.read_csv(CORE50_PATH)

core50_genes = (
    core50_df["Gene"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

assert len(core50_genes) == 50
assert "FOXM1" in core50_genes

core50_lto_foxm1 = [
    g for g in core50_genes
    if g != "FOXM1"
]

assert len(core50_lto_foxm1) == 49

# --------------------------------------------------
# Step31 frozen independent programs
# --------------------------------------------------

program_map_df = pd.read_csv(PROGRAM_MAP_PATH)

retained_map = (
    program_map_df.loc[
        program_map_df["mapping_status"] == "retained"
    ]
    .copy()
)

def frozen_program(name):
    return (
        retained_map.loc[
            retained_map["program"] == name,
            "requested_symbol"
        ]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

g2m_genes = frozen_program(
    "G2M_Independent"
)

spindle_genes = frozen_program(
    "Spindle_Chromosome_Independent"
)

mitotic_genes = frozen_program(
    "Mitotic_Entry_Independent"
)

hepatic_genes = frozen_program(
    "Hepatic_Differentiation"
)

bile_genes = frozen_program(
    "Bile_Acid_Metabolism"
)

# --------------------------------------------------
# Exact frozen counts
# --------------------------------------------------

assert len(g2m_genes) == 22
assert len(spindle_genes) == 29
assert len(mitotic_genes) == 16
assert len(hepatic_genes) == 19
assert len(bile_genes) == 18

ENDPOINTS = {
    # Primary
    "Core50_LTO_FOXM1": core50_lto_foxm1,
    "G2M_Independent": g2m_genes,
    "Mitotic_Entry_Independent": mitotic_genes,
    "Spindle_Chromosome_Independent": spindle_genes,

    # Secondary directional
    "Hepatic_Differentiation": hepatic_genes,
    "Bile_Acid_Metabolism": bile_genes,

    # Secondary descriptive only
    "Core50_Full": core50_genes,
}

endpoint_roles = {
    "Core50_LTO_FOXM1": "primary",
    "G2M_Independent": "primary",
    "Mitotic_Entry_Independent": "primary",
    "Spindle_Chromosome_Independent": "primary",
    "Hepatic_Differentiation": "secondary_directional",
    "Bile_Acid_Metabolism": "secondary_directional",
    "Core50_Full": "secondary_descriptive",
}

print("Frozen endpoint counts:")
for name, genes in ENDPOINTS.items():
    print(
        f"{name:35s}",
        len(genes),
        endpoint_roles[name]
    )

# --------------------------------------------------
# Save long-format frozen lock
# --------------------------------------------------

rows = []

for endpoint, genes in ENDPOINTS.items():
    for rank, gene in enumerate(genes, start=1):
        rows.append({
            "endpoint": endpoint,
            "role": endpoint_roles[endpoint],
            "rank_within_endpoint": rank,
            "gene": gene
        })

endpoint_lock_df = pd.DataFrame(rows)

LOCK_PATH = (
    OUT /
    "Step33_FROZEN_ENDPOINT_GENESETS.csv"
)

endpoint_lock_df.to_csv(
    LOCK_PATH,
    index=False
)

print()
print("Saved:")
print(LOCK_PATH)

print("=" * 70)
print("Frozen endpoint lock: PASS")

In [ ]:
# Cell 41 — Endpoint coverage and MS3 population audit

import numpy as np
import pandas as pd

print("=" * 70)
print("Cell 41 — Endpoint coverage + MS3 audit")
print("=" * 70)

fit_gene_set = set(
    oracle_final.adata.var_names.astype(str)
)

coverage_rows = []

for endpoint, genes in ENDPOINTS.items():

    present = [
        g for g in genes
        if g in fit_gene_set
    ]

    missing = [
        g for g in genes
        if g not in fit_gene_set
    ]

    coverage_rows.append({
        "endpoint": endpoint,
        "requested_n": len(genes),
        "present_n": len(present),
        "missing_n": len(missing),
        "coverage_fraction":
            len(present) / len(genes),
        "missing_genes":
            ";".join(missing)
    })

coverage_df = pd.DataFrame(
    coverage_rows
)

print("\nEndpoint coverage:")
print(
    coverage_df[
        [
            "endpoint",
            "requested_n",
            "present_n",
            "missing_n",
            "coverage_fraction"
        ]
    ].to_string(index=False)
)

print()

for _, row in coverage_df.iterrows():

    if row["missing_n"] > 0:

        print(
            f'{row["endpoint"]} missing: '
            f'{row["missing_genes"]}'
        )

# --------------------------------------------------
# MS3 biological evaluation population
# --------------------------------------------------

state_key = "final_malignant_state_id"
patient_key = "patient"

assert state_key in oracle_final.adata.obs.columns
assert patient_key in oracle_final.adata.obs.columns

ms3_mask = (
    oracle_final.adata.obs[state_key]
    .astype(str)
    .values
    ==
    "MS3"
)

ms3_n = int(
    ms3_mask.sum()
)

ms3_patients = (
    oracle_final.adata.obs.loc[
        ms3_mask,
        patient_key
    ]
    .astype(str)
)

print()
print("MS3 cells:", ms3_n)
print("MS3 patients:", ms3_patients.nunique())

print()
print("MS3 cells by patient:")
print(
    ms3_patients
    .value_counts()
    .sort_index()
    .to_string()
)

# --------------------------------------------------
# Save audit
# --------------------------------------------------

COVERAGE_PATH = (
    OUT /
    "Step33_FROZEN_ENDPOINT_COVERAGE.csv"
)

coverage_df.to_csv(
    COVERAGE_PATH,
    index=False
)

all_full = bool(
    (coverage_df["missing_n"] == 0).all()
)

print()
print(
    "ALL ENDPOINTS FULLY COVERED:",
    all_full
)

print()
print("Saved:")
print(COVERAGE_PATH)

print("=" * 70)

In [ ]:
# Cell 41B — Frozen symbol alias + true network absence audit

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("Cell 41B — Alias + true network absence audit")
print("=" * 70)

CORE50_MAP_PATH = Path(
    "/mnt/e/TCGA/LIHC_scRNA_Step31_Independent_Validation/"
    "Audit/Step31_Core50_FrozenValidationMapping.csv"
)

core50_map = pd.read_csv(CORE50_MAP_PATH)

# --------------------------------------------------
# Missing genes from Cell 41
# --------------------------------------------------

missing_by_endpoint = {}

for _, row in coverage_df.iterrows():

    if int(row["missing_n"]) > 0:

        missing_by_endpoint[row["endpoint"]] = (
            str(row["missing_genes"])
            .split(";")
        )

all_missing = sorted(
    set(
        g
        for genes in missing_by_endpoint.values()
        for g in genes
    )
)

print("\nAll nominally missing symbols:")
print(all_missing)

# --------------------------------------------------
# Step31 frozen Core50 mapping schema
# --------------------------------------------------

print("\n" + "-" * 70)
print("[Step31 Core50 frozen mapping]")
print("Shape:", core50_map.shape)
print("Columns:", core50_map.columns.tolist())

print()

# Print rows mentioning currently missing Core50 symbols
mask = np.zeros(len(core50_map), dtype=bool)

for col in core50_map.columns:

    s = core50_map[col].astype(str)

    for gene in missing_by_endpoint.get(
        "Core50_LTO_FOXM1", []
    ):
        mask |= s.str.upper().eq(gene.upper()).values

core_missing_map = core50_map.loc[mask].copy()

print(core_missing_map.to_string(index=False))

# --------------------------------------------------
# Step31 program mapping rows for missing genes
# --------------------------------------------------

program_missing = sorted(
    set(
        missing_by_endpoint.get("G2M_Independent", [])
        +
        missing_by_endpoint.get(
            "Spindle_Chromosome_Independent", []
        )
        +
        missing_by_endpoint.get(
            "Bile_Acid_Metabolism", []
        )
    )
)

print("\n" + "-" * 70)
print("[Step31 program mapping — missing requested symbols]")

print(
    program_map_df.loc[
        program_map_df["requested_symbol"]
        .isin(program_missing)
    ].to_string(index=False)
)

# --------------------------------------------------
# Inspect current CellOracle var metadata
# --------------------------------------------------

print("\n" + "-" * 70)
print("[CellOracle var metadata]")

print(
    "var columns:",
    oracle_final.adata.var.columns.tolist()
)

print()
print(
    oracle_final.adata.var.head().to_string()
)

# --------------------------------------------------
# Try Ensembl-ID recovery from var metadata
# --------------------------------------------------

id_cols = [
    c for c in oracle_final.adata.var.columns
    if (
        "ensembl" in c.lower()
        or "gene_id" in c.lower()
        or c.lower() == "id"
    )
]

print()
print("Candidate ID columns:", id_cols)

recovery_rows = []

missing_program_rows = (
    program_map_df.loc[
        program_map_df["requested_symbol"]
        .isin(program_missing)
    ]
    .copy()
)

for _, r in missing_program_rows.iterrows():

    requested = str(r["requested_symbol"])
    ensembl = str(r["selected_ensembl_id"]).split(".")[0]

    recovered_symbols = []

    for col in id_cols:

        values = (
            oracle_final.adata.var[col]
            .astype(str)
            .str.split(".")
            .str[0]
        )

        hit = values == ensembl

        if hit.any():

            recovered_symbols.extend(
                oracle_final.adata.var_names[
                    hit.values
                ].astype(str).tolist()
            )

    recovery_rows.append({
        "requested_symbol": requested,
        "selected_ensembl_id": ensembl,
        "recovered_current_symbols":
            ";".join(sorted(set(recovered_symbols))),
        "recovered_n":
            len(set(recovered_symbols))
    })

recovery_df = pd.DataFrame(recovery_rows)

print("\n" + "-" * 70)
print("[Ensembl-based recovery audit]")

if len(recovery_df) > 0:
    print(recovery_df.to_string(index=False))
else:
    print("No missing program rows to audit.")

print("=" * 70)

In [ ]:
# Cell 42 — Frozen endpoint projection + alias lock

import pandas as pd
import json

print("=" * 70)
print("Cell 42 — Frozen endpoint projection + alias lock")
print("=" * 70)

fit_gene_set = set(
    oracle_final.adata.var_names.astype(str)
)

# --------------------------------------------------
# Frozen aliases from Step31
# --------------------------------------------------

FROZEN_ALIAS_MAP = {
    "SGOL1": "SGO1",
    "SGOL2": "SGO2",
}

projection_rows = []
SIM_ENDPOINTS = {}

for endpoint, genes in ENDPOINTS.items():

    projected = []

    for gene in genes:

        if gene in fit_gene_set:

            sim_gene = gene
            status = "direct"

        elif (
            gene in FROZEN_ALIAS_MAP
            and FROZEN_ALIAS_MAP[gene] in fit_gene_set
        ):

            sim_gene = FROZEN_ALIAS_MAP[gene]
            status = "frozen_alias"

        else:

            sim_gene = None
            status = "structurally_unavailable"

        projection_rows.append({
            "endpoint": endpoint,
            "canonical_gene": gene,
            "simulation_gene": sim_gene,
            "status": status
        })

        if sim_gene is not None:
            projected.append(sim_gene)

    # preserve order, remove accidental duplicates
    projected = list(dict.fromkeys(projected))

    SIM_ENDPOINTS[endpoint] = projected


projection_df = pd.DataFrame(
    projection_rows
)

summary_df = (
    projection_df
    .groupby(
        ["endpoint", "status"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print("\nProjection summary:")
print(summary_df.to_string(index=False))

print("\nFinal simulatable endpoint sizes:")

for endpoint, genes in SIM_ENDPOINTS.items():
    print(
        f"{endpoint:35s}",
        len(genes)
    )

print("\nAlias recoveries:")

alias_rows = projection_df.loc[
    projection_df["status"] == "frozen_alias"
]

if len(alias_rows):
    print(alias_rows.to_string(index=False))
else:
    print("None")

# --------------------------------------------------
# Save
# --------------------------------------------------

PROJECTION_PATH = (
    OUT /
    "Step33_FROZEN_ENDPOINT_CELLORACLE_PROJECTION.csv"
)

projection_df.to_csv(
    PROJECTION_PATH,
    index=False
)

print()
print("Saved:")
print(PROJECTION_PATH)

print("=" * 70)

In [ ]:
# Cell 43 — Baseline-Anchored CellOracle Perturbation Score

import numpy as np
import pandas as pd
import scipy.sparse as sp

print("=" * 70)
print("Cell 43 — Baseline-Anchored Perturbation Scoring")
print("=" * 70)

# --------------------------------------------------
# Union of all simulatable endpoint genes
# --------------------------------------------------

all_score_genes = sorted(
    set(
        g
        for genes in SIM_ENDPOINTS.values()
        for g in genes
    )
)

print(
    "Unique endpoint genes used:",
    len(all_score_genes)
)

gene_to_idx = {
    g: oracle_final.adata.var_names.get_loc(g)
    for g in all_score_genes
}

idx = [
    gene_to_idx[g]
    for g in all_score_genes
]

# --------------------------------------------------
# Extract only endpoint genes
# Avoid copying whole 12057 x 10776 matrix
# --------------------------------------------------

baseline = (
    oracle_final.adata.layers[
        "imputed_count"
    ][:, idx]
)

simulated = (
    oracle_final.adata.layers[
        "simulated_count"
    ][:, idx]
)

if sp.issparse(baseline):
    baseline = baseline.toarray()

if sp.issparse(simulated):
    simulated = simulated.toarray()

baseline = np.asarray(
    baseline,
    dtype=np.float32
)

simulated = np.asarray(
    simulated,
    dtype=np.float32
)

print(
    "Scoring matrix:",
    baseline.shape
)

# --------------------------------------------------
# Baseline-only normalization
# Frozen method
# --------------------------------------------------

mu = baseline.mean(
    axis=0,
    dtype=np.float64
)

sigma = baseline.std(
    axis=0,
    ddof=0,
    dtype=np.float64
)

zero_sd = np.where(
    sigma <= 1e-12
)[0]

print(
    "Zero-variance genes:",
    len(zero_sd)
)

if len(zero_sd) > 0:

    print(
        [
            all_score_genes[i]
            for i in zero_sd
        ]
    )

assert len(zero_sd) == 0, (
    "Zero-variance endpoint genes detected. "
    "Stop scoring and review."
)

# Same baseline mu/sigma for BOTH matrices
z_base = (
    baseline - mu
) / sigma

z_sim = (
    simulated - mu
) / sigma

print(
    "Baseline anchoring: PASS"
)

# --------------------------------------------------
# MS3 population
# --------------------------------------------------

state_key = "final_malignant_state_id"
patient_key = "patient"

ms3_mask = (
    oracle_final.adata.obs[
        state_key
    ]
    .astype(str)
    .values
    ==
    "MS3"
)

assert int(ms3_mask.sum()) == 720

ms3_obs = (
    oracle_final.adata.obs.loc[
        ms3_mask,
        [patient_key, state_key]
    ]
    .copy()
)

# --------------------------------------------------
# Cell-level scoring
# --------------------------------------------------

cell_scores = pd.DataFrame(
    index=oracle_final.adata.obs_names[
        ms3_mask
    ]
)

cell_scores["patient"] = (
    ms3_obs[patient_key]
    .astype(str)
    .values
)

cell_scores["state"] = "MS3"

for endpoint, genes in SIM_ENDPOINTS.items():

    endpoint_idx = [
        all_score_genes.index(g)
        for g in genes
    ]

    base_score = (
        z_base[
            ms3_mask
        ][:, endpoint_idx]
        .mean(axis=1)
    )

    sim_score = (
        z_sim[
            ms3_mask
        ][:, endpoint_idx]
        .mean(axis=1)
    )

    delta = (
        sim_score
        -
        base_score
    )

    cell_scores[
        f"{endpoint}__baseline"
    ] = base_score

    cell_scores[
        f"{endpoint}__simulated"
    ] = sim_score

    cell_scores[
        f"{endpoint}__delta"
    ] = delta

# --------------------------------------------------
# Save
# --------------------------------------------------

CELL_SCORE_PATH = (
    OUT /
    "Step33_FOXM1_MS3_CellLevel_BaselineAnchoredScores.csv"
)

cell_scores.to_csv(
    CELL_SCORE_PATH
)

print()
print("MS3 cells scored:", len(cell_scores))

print()
print("Saved:")
print(CELL_SCORE_PATH)

print("=" * 70)
print("Baseline-anchored scoring: COMPLETE")

In [ ]:
# Cell 44 — MS3 overall descriptive summary

print("=" * 70)
print("Cell 44 — MS3 overall FOXM1 perturbation summary")
print("=" * 70)

summary_rows = []

for endpoint, genes in SIM_ENDPOINTS.items():

    x = cell_scores[
        f"{endpoint}__delta"
    ].values

    summary_rows.append({
        "endpoint": endpoint,
        "simulatable_genes": len(genes),

        "median_delta":
            float(np.median(x)),

        "Q1_delta":
            float(np.quantile(x, 0.25)),

        "Q3_delta":
            float(np.quantile(x, 0.75)),

        "mean_delta":
            float(np.mean(x)),

        "fraction_negative":
            float(np.mean(x < 0)),

        "fraction_positive":
            float(np.mean(x > 0)),
    })

overall_summary = pd.DataFrame(
    summary_rows
)

print(
    overall_summary.to_string(
        index=False
    )
)

SUMMARY_PATH = (
    OUT /
    "Step33_FOXM1_MS3_Overall_ProgramDelta.csv"
)

overall_summary.to_csv(
    SUMMARY_PATH,
    index=False
)

print()
print("Saved:")
print(SUMMARY_PATH)

print("=" * 70)

In [ ]:
# Cell 45 — Patient-level FOXM1 perturbation consistency

print("=" * 70)
print("Cell 45 — Patient-level FOXM1 perturbation consistency")
print("=" * 70)

PRIMARY_PATIENTS = [
    "HCC08",
    "HCC09",
    "HCC10"
]

SPARSE_PATIENTS = [
    "HCC04",
    "HCC07"
]

EXPECTED_DIRECTION = {
    "Core50_LTO_FOXM1": "negative",
    "G2M_Independent": "negative",
    "Mitotic_Entry_Independent": "negative",
    "Spindle_Chromosome_Independent": "negative",

    "Hepatic_Differentiation": "positive",
    "Bile_Acid_Metabolism": "positive",

    "Core50_Full": "negative",
}

patient_rows = []

for endpoint in SIM_ENDPOINTS.keys():

    delta_col = (
        f"{endpoint}__delta"
    )

    for patient, sub in cell_scores.groupby(
        "patient"
    ):

        x = sub[
            delta_col
        ].values

        patient_rows.append({
            "endpoint": endpoint,
            "patient": patient,
            "n_cells": len(sub),

            "median_delta":
                float(np.median(x)),

            "Q1_delta":
                float(np.quantile(x, 0.25)),

            "Q3_delta":
                float(np.quantile(x, 0.75)),

            "mean_delta":
                float(np.mean(x)),

            "patient_role":
                (
                    "primary_replication"
                    if patient in PRIMARY_PATIENTS
                    else
                    "sparse_sensitivity"
                )
        })

patient_summary = pd.DataFrame(
    patient_rows
)

print("\nPatient-level medians:")
print(
    patient_summary[
        [
            "endpoint",
            "patient",
            "n_cells",
            "median_delta",
            "patient_role"
        ]
    ].to_string(
        index=False
    )
)

# --------------------------------------------------
# Primary replication direction consistency
# --------------------------------------------------

direction_rows = []

for endpoint in SIM_ENDPOINTS.keys():

    sub = patient_summary.loc[
        (
            patient_summary["endpoint"] == endpoint
        )
        &
        (
            patient_summary["patient"]
            .isin(PRIMARY_PATIENTS)
        )
    ].copy()

    expected = EXPECTED_DIRECTION[
        endpoint
    ]

    if expected == "negative":

        concordant = (
            sub["median_delta"] < 0
        )

    else:

        concordant = (
            sub["median_delta"] > 0
        )

    direction_rows.append({
        "endpoint": endpoint,
        "expected_direction": expected,
        "primary_patients_n": len(sub),
        "concordant_patients":
            int(concordant.sum()),
        "direction_consistency":
            f"{int(concordant.sum())}/{len(sub)}"
    })

direction_summary = pd.DataFrame(
    direction_rows
)

print()
print("Primary patient direction consistency:")
print(
    direction_summary.to_string(
        index=False
    )
)

# --------------------------------------------------
# Save
# --------------------------------------------------

PATIENT_PATH = (
    OUT /
    "Step33_FOXM1_MS3_PatientLevel_ProgramDelta.csv"
)

DIRECTION_PATH = (
    OUT /
    "Step33_FOXM1_MS3_PrimaryPatient_DirectionConsistency.csv"
)

patient_summary.to_csv(
    PATIENT_PATH,
    index=False
)

direction_summary.to_csv(
    DIRECTION_PATH,
    index=False
)

print()
print("Saved:")
print(PATIENT_PATH)
print(DIRECTION_PATH)

print("=" * 70)

In [ ]:
# Cell 46 — FOXM1 response topology audit

import numpy as np
import pandas as pd

print("=" * 70)
print("Cell 46 — FOXM1 response topology audit")
print("=" * 70)

# Frozen FOXM1 direct targets in simulation topology
foxm1_direct_targets = set(
    grn_sim.loc[
        grn_sim["TF"] == "FOXM1",
        "target"
    ].astype(str)
)

rows = []

for endpoint, genes in SIM_ENDPOINTS.items():

    # endpoint gene indices in scoring matrix
    idx_ep = [
        all_score_genes.index(g)
        for g in genes
    ]

    # MS3 gene-level delta
    gene_delta = (
        z_sim[ms3_mask][:, idx_ep]
        -
        z_base[ms3_mask][:, idx_ep]
    )

    # median across MS3 cells for each gene
    gene_median_delta = np.median(
        gene_delta,
        axis=0
    )

    nonzero_mask = (
        np.abs(gene_median_delta) > 1e-12
    )

    direct_overlap = [
        g for g in genes
        if g in foxm1_direct_targets
    ]

    rows.append({
        "endpoint": endpoint,
        "simulatable_genes": len(genes),

        "genes_with_nonzero_median_delta":
            int(nonzero_mask.sum()),

        "fraction_nonzero":
            float(nonzero_mask.mean()),

        "FOXM1_direct_target_overlap_n":
            len(direct_overlap),

        "FOXM1_direct_targets":
            ";".join(direct_overlap),

        "median_gene_delta":
            float(np.median(gene_median_delta)),

        "min_gene_delta":
            float(np.min(gene_median_delta)),

        "max_gene_delta":
            float(np.max(gene_median_delta)),
    })

response_topology_df = pd.DataFrame(rows)

print(
    response_topology_df.to_string(
        index=False
    )
)

TOPOLOGY_PATH = (
    OUT /
    "Step33_FOXM1_Endpoint_ResponseTopology.csv"
)

response_topology_df.to_csv(
    TOPOLOGY_PATH,
    index=False
)

print()
print("Saved:")
print(TOPOLOGY_PATH)

print("=" * 70)

In [ ]:
# Cell 47 — FOXM1 pilot result lock

import json

print("=" * 70)
print("Cell 47 — FOXM1 PILOT RESULT LOCK")
print("=" * 70)

def get_overall(endpoint):
    r = overall_summary.loc[
        overall_summary["endpoint"] == endpoint
    ].iloc[0]

    return {
        "simulatable_genes":
            int(r["simulatable_genes"]),
        "median_delta":
            float(r["median_delta"]),
        "Q1_delta":
            float(r["Q1_delta"]),
        "Q3_delta":
            float(r["Q3_delta"]),
        "fraction_negative":
            float(r["fraction_negative"]),
    }

def get_consistency(endpoint):
    r = direction_summary.loc[
        direction_summary["endpoint"] == endpoint
    ].iloc[0]

    return str(
        r["direction_consistency"]
    )

FOXM1_LOCK = {
    "status": "FOXM1_PILOT_FROZEN",

    "perturbation": {
        "TF": "FOXM1",
        "n_propagation": 3,
        "use_randomized_GRN": False,
        "clip_delta_X": False
    },

    "primary_population": {
        "state": "MS3",
        "cells": 720,
        "primary_replication_patients": [
            "HCC08",
            "HCC09",
            "HCC10"
        ],
        "sparse_sensitivity_patients": [
            "HCC04",
            "HCC07"
        ]
    },

    "primary_endpoints": {
        "Core50_LTO_FOXM1":
            get_overall("Core50_LTO_FOXM1"),

        "G2M_Independent":
            get_overall("G2M_Independent"),

        "Mitotic_Entry_Independent":
            get_overall("Mitotic_Entry_Independent"),

        "Spindle_Chromosome_Independent":
            get_overall("Spindle_Chromosome_Independent"),
    },

    "primary_patient_direction_consistency": {
        "Core50_LTO_FOXM1":
            get_consistency("Core50_LTO_FOXM1"),

        "G2M_Independent":
            get_consistency("G2M_Independent"),

        "Mitotic_Entry_Independent":
            get_consistency("Mitotic_Entry_Independent"),

        "Spindle_Chromosome_Independent":
            get_consistency(
                "Spindle_Chromosome_Independent"
            ),
    },

    "secondary_directional": {
        "Hepatic_Differentiation":
            get_overall("Hepatic_Differentiation"),

        "Bile_Acid_Metabolism":
            get_overall("Bile_Acid_Metabolism"),
    },

    "interpretation_lock":
        (
            "FOXM1 virtual suppression predicts weakening of the "
            "conserved MS3 proliferative/mitotic transcriptional "
            "program across the three primary replication patients. "
            "No predicted restoration of hepatic differentiation or "
            "bile-acid metabolic programs was observed."
        ),

    "claim_boundary": (
        "In silico counterfactual compatibility only; "
        "not causal validation, therapeutic efficacy, "
        "or lineage conversion."
    )
}

LOCK_PATH = (
    OUT /
    "Step33_FOXM1_PILOT_FINAL_LOCK.json"
)

with open(
    LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        FOXM1_LOCK,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    FOXM1_LOCK,
    indent=2,
    ensure_ascii=False
))

print()
print("Saved:")
print(LOCK_PATH)

print("=" * 70)
print("FOXM1 PILOT: FROZEN")
print("=" * 70)

In [ ]:
# Cell 46B — FOXM1 gene-level response lock

print("=" * 70)
print("Cell 46B — FOXM1 gene-level response lock")
print("=" * 70)

gene_response_rows = []

for endpoint, genes in SIM_ENDPOINTS.items():

    idx_ep = [
        all_score_genes.index(g)
        for g in genes
    ]

    gene_delta = (
        z_sim[ms3_mask][:, idx_ep]
        -
        z_base[ms3_mask][:, idx_ep]
    )

    med = np.median(
        gene_delta,
        axis=0
    )

    for gene, delta in zip(
        genes,
        med
    ):

        if delta < -1e-12:
            direction = "decreased"
        elif delta > 1e-12:
            direction = "increased"
        else:
            direction = "unchanged"

        gene_response_rows.append({
            "endpoint": endpoint,
            "gene": gene,
            "median_delta": float(delta),
            "direction": direction,
            "FOXM1_direct_target":
                gene in foxm1_direct_targets
        })

gene_response_df = pd.DataFrame(
    gene_response_rows
)

# Focus on primary endpoints
primary_names = [
    "Core50_LTO_FOXM1",
    "G2M_Independent",
    "Mitotic_Entry_Independent",
    "Spindle_Chromosome_Independent"
]

primary_gene_response = (
    gene_response_df.loc[
        gene_response_df["endpoint"]
        .isin(primary_names)
    ]
    .copy()
)

print("\nNon-zero primary endpoint responses:")

print(
    primary_gene_response.loc[
        primary_gene_response["direction"]
        != "unchanged"
    ]
    .sort_values(
        ["endpoint", "median_delta"]
    )
    .to_string(index=False)
)

GENE_RESPONSE_PATH = (
    OUT /
    "Step33_FOXM1_MS3_GeneLevel_Response.csv"
)

gene_response_df.to_csv(
    GENE_RESPONSE_PATH,
    index=False
)

print()
print("Saved:")
print(GENE_RESPONSE_PATH)

print("=" * 70)

In [ ]:
# Cell 48 — MYBL2 endpoint membership + virtual knockout

print("=" * 70)
print("Cell 48 — MYBL2 virtual knockout")
print("=" * 70)

KO_TF = "MYBL2"

print("Endpoint membership:")

for endpoint, genes in ENDPOINTS.items():
    print(
        f"{endpoint:35s}",
        KO_TF in genes
    )

assert KO_TF in oracle_final.all_regulatory_genes_in_TFdict
assert KO_TF in oracle_final.active_regulatory_genes
assert KO_TF in oracle_final.high_var_genes

oracle_final.simulate_shift(
    perturb_condition={
        KO_TF: 0.0
    },
    n_propagation=3,
    use_randomized_GRN=False,
    clip_delta_X=False
)

print()
print("MYBL2 simulation completed.")

print("Layers:")
print(
    list(
        oracle_final.adata.layers.keys()
    )
)

print("=" * 70)

In [ ]:
# Cell 49 — MYBL2 simulation QC

print("=" * 70)
print("Cell 49 — MYBL2 simulation QC")
print("=" * 70)

gene_idx = (
    oracle_final.adata.var_names
    .get_loc("MYBL2")
)

mybl2_base = get_gene_vector(
    "imputed_count",
    gene_idx
)

mybl2_sim = get_gene_vector(
    "simulated_count",
    gene_idx
)

mybl2_delta = (
    mybl2_sim
    -
    mybl2_base
)

print(
    "Baseline median:",
    float(np.median(mybl2_base))
)

print(
    "Simulated median:",
    float(np.median(mybl2_sim))
)

print(
    "Delta median:",
    float(np.median(mybl2_delta))
)

print(
    "Simulated min/max:",
    float(np.min(mybl2_sim)),
    float(np.max(mybl2_sim))
)

oracle_final.evaluate_simulated_gene_distribution_range()

print()
print("Distribution-range QC completed.")

print("=" * 70)

In [ ]:
# Cell 50 — MYBL2-specific leave-target-out endpoint projection

print("=" * 70)
print("Cell 50 — MYBL2 endpoint projection")
print("=" * 70)

MYBL2_ENDPOINTS = {}

for endpoint, genes in SIM_ENDPOINTS.items():

    # Remove perturbed TF only if it belongs to endpoint
    adjusted = [
        g for g in genes
        if g != "MYBL2"
    ]

    MYBL2_ENDPOINTS[
        endpoint.replace(
            "LTO_FOXM1",
            "LTO_MYBL2"
        )
    ] = adjusted

    removed = (
        len(genes) - len(adjusted)
    )

    print(
        f"{endpoint:35s}",
        "original =", len(genes),
        "| MYBL2 removed =", removed,
        "| final =", len(adjusted)
    )

print("=" * 70)

# Cell 50–53 — MYBL2 Formal Perturbation Scoring

Use the same frozen baseline-anchored scoring framework as FOXM1.

- Baseline mean/SD remain fixed from `imputed_count`
- Current `simulated_count` corresponds to MYBL2 KO
- Leave-target-out is applied automatically if MYBL2 belongs to an endpoint
- Primary biological replication: HCC08, HCC09, HCC10
- HCC04/HCC07 remain sparse-cell sensitivity only

In [ ]:
# Cell 50 — MYBL2-specific endpoint projection + leave-target-out

print("=" * 70)
print("Cell 50 — MYBL2 endpoint projection")
print("=" * 70)

KO_TF = "MYBL2"

# Start from network-covered frozen programs.
# Core50_Full is the common source for TF-specific Core50 LTO.

BASE_SIM_ENDPOINTS = {
    "Core50": SIM_ENDPOINTS["Core50_Full"],
    "G2M_Independent": SIM_ENDPOINTS["G2M_Independent"],
    "Mitotic_Entry_Independent":
        SIM_ENDPOINTS["Mitotic_Entry_Independent"],
    "Spindle_Chromosome_Independent":
        SIM_ENDPOINTS["Spindle_Chromosome_Independent"],
    "Hepatic_Differentiation":
        SIM_ENDPOINTS["Hepatic_Differentiation"],
    "Bile_Acid_Metabolism":
        SIM_ENDPOINTS["Bile_Acid_Metabolism"],
}

MYBL2_ENDPOINTS = {}

for endpoint, genes in BASE_SIM_ENDPOINTS.items():

    adjusted = [
        g for g in genes
        if g != KO_TF
    ]

    if endpoint == "Core50":
        out_name = "Core50_LTO_MYBL2"
    else:
        out_name = endpoint

    MYBL2_ENDPOINTS[out_name] = adjusted

    print(
        f"{out_name:35s}",
        "requested =", len(genes),
        "| MYBL2 present =", KO_TF in genes,
        "| final =", len(adjusted)
    )

print("=" * 70)

In [ ]:
# Cell 51 — MYBL2 baseline-anchored perturbation scoring

import numpy as np
import pandas as pd
import scipy.sparse as sp

print("=" * 70)
print("Cell 51 — MYBL2 baseline-anchored scoring")
print("=" * 70)

# --------------------------------------------------
# Current endpoint union
# --------------------------------------------------

mybl2_score_genes = sorted(
    set(
        g
        for genes in MYBL2_ENDPOINTS.values()
        for g in genes
    )
)

mybl2_idx = [
    oracle_final.adata.var_names.get_loc(g)
    for g in mybl2_score_genes
]

# --------------------------------------------------
# Extract baseline + CURRENT MYBL2 simulation
# --------------------------------------------------

base_now = (
    oracle_final.adata.layers[
        "imputed_count"
    ][:, mybl2_idx]
)

sim_now = (
    oracle_final.adata.layers[
        "simulated_count"
    ][:, mybl2_idx]
)

if sp.issparse(base_now):
    base_now = base_now.toarray()

if sp.issparse(sim_now):
    sim_now = sim_now.toarray()

base_now = np.asarray(
    base_now,
    dtype=np.float32
)

sim_now = np.asarray(
    sim_now,
    dtype=np.float32
)

# --------------------------------------------------
# Baseline-only μ / σ
# --------------------------------------------------

mu_mybl2 = base_now.mean(
    axis=0,
    dtype=np.float64
)

sigma_mybl2 = base_now.std(
    axis=0,
    ddof=0,
    dtype=np.float64
)

assert np.all(
    sigma_mybl2 > 1e-12
)

z_base_mybl2 = (
    base_now - mu_mybl2
) / sigma_mybl2

z_sim_mybl2 = (
    sim_now - mu_mybl2
) / sigma_mybl2

# --------------------------------------------------
# MS3 cells
# --------------------------------------------------

ms3_mask = (
    oracle_final.adata.obs[
        "final_malignant_state_id"
    ]
    .astype(str)
    .values
    ==
    "MS3"
)

assert int(ms3_mask.sum()) == 720

mybl2_cell_scores = pd.DataFrame(
    index=oracle_final.adata.obs_names[
        ms3_mask
    ]
)

mybl2_cell_scores["patient"] = (
    oracle_final.adata.obs.loc[
        ms3_mask,
        "patient"
    ]
    .astype(str)
    .values
)

for endpoint, genes in MYBL2_ENDPOINTS.items():

    idx_ep = [
        mybl2_score_genes.index(g)
        for g in genes
    ]

    baseline_score = (
        z_base_mybl2[
            ms3_mask
        ][:, idx_ep]
        .mean(axis=1)
    )

    simulated_score = (
        z_sim_mybl2[
            ms3_mask
        ][:, idx_ep]
        .mean(axis=1)
    )

    mybl2_cell_scores[
        f"{endpoint}__baseline"
    ] = baseline_score

    mybl2_cell_scores[
        f"{endpoint}__simulated"
    ] = simulated_score

    mybl2_cell_scores[
        f"{endpoint}__delta"
    ] = (
        simulated_score
        -
        baseline_score
    )

MYBL2_CELL_PATH = (
    OUT /
    "Step33_MYBL2_MS3_CellLevel_BaselineAnchoredScores.csv"
)

mybl2_cell_scores.to_csv(
    MYBL2_CELL_PATH
)

print(
    "MS3 cells scored:",
    len(mybl2_cell_scores)
)

print("Saved:")
print(MYBL2_CELL_PATH)

print("=" * 70)

In [ ]:
# Cell 52 — MYBL2 overall and patient-level results

print("=" * 70)
print("Cell 52 — MYBL2 perturbation results")
print("=" * 70)

PRIMARY_PATIENTS = [
    "HCC08",
    "HCC09",
    "HCC10"
]

overall_rows = []
patient_rows = []

for endpoint, genes in MYBL2_ENDPOINTS.items():

    delta_col = (
        f"{endpoint}__delta"
    )

    x = mybl2_cell_scores[
        delta_col
    ].values

    overall_rows.append({
        "endpoint": endpoint,
        "simulatable_genes": len(genes),
        "median_delta":
            float(np.median(x)),
        "Q1_delta":
            float(np.quantile(x, 0.25)),
        "Q3_delta":
            float(np.quantile(x, 0.75)),
        "mean_delta":
            float(np.mean(x)),
        "fraction_negative":
            float(np.mean(x < 0)),
        "fraction_positive":
            float(np.mean(x > 0)),
    })

    for patient, sub in (
        mybl2_cell_scores
        .groupby("patient")
    ):

        px = sub[
            delta_col
        ].values

        patient_rows.append({
            "endpoint": endpoint,
            "patient": patient,
            "n_cells": len(sub),
            "median_delta":
                float(np.median(px)),
            "Q1_delta":
                float(np.quantile(px, 0.25)),
            "Q3_delta":
                float(np.quantile(px, 0.75)),
            "patient_role":
                (
                    "primary_replication"
                    if patient in PRIMARY_PATIENTS
                    else
                    "sparse_sensitivity"
                )
        })

mybl2_overall = pd.DataFrame(
    overall_rows
)

mybl2_patient = pd.DataFrame(
    patient_rows
)

print("\nOverall:")
print(
    mybl2_overall.to_string(
        index=False
    )
)

print("\nPatient medians:")
print(
    mybl2_patient[
        [
            "endpoint",
            "patient",
            "n_cells",
            "median_delta",
            "patient_role"
        ]
    ].to_string(
        index=False
    )
)

# --------------------------------------------------
# Direction consistency
# --------------------------------------------------

EXPECTED_DIRECTION = {
    "Core50_LTO_MYBL2": "negative",
    "G2M_Independent": "negative",
    "Mitotic_Entry_Independent": "negative",
    "Spindle_Chromosome_Independent": "negative",
    "Hepatic_Differentiation": "positive",
    "Bile_Acid_Metabolism": "positive",
}

direction_rows = []

for endpoint in MYBL2_ENDPOINTS.keys():

    sub = mybl2_patient.loc[
        (
            mybl2_patient["endpoint"]
            == endpoint
        )
        &
        (
            mybl2_patient["patient"]
            .isin(PRIMARY_PATIENTS)
        )
    ]

    expected = EXPECTED_DIRECTION[
        endpoint
    ]

    if expected == "negative":
        concordant = (
            sub["median_delta"] < 0
        )
    else:
        concordant = (
            sub["median_delta"] > 0
        )

    direction_rows.append({
        "endpoint": endpoint,
        "expected_direction": expected,
        "primary_patients_n":
            len(sub),
        "concordant_patients":
            int(concordant.sum()),
        "direction_consistency":
            f"{int(concordant.sum())}/{len(sub)}"
    })

mybl2_direction = pd.DataFrame(
    direction_rows
)

print()
print("Primary patient direction consistency:")
print(
    mybl2_direction.to_string(
        index=False
    )
)

# Save

mybl2_overall.to_csv(
    OUT /
    "Step33_MYBL2_MS3_Overall_ProgramDelta.csv",
    index=False
)

mybl2_patient.to_csv(
    OUT /
    "Step33_MYBL2_MS3_PatientLevel_ProgramDelta.csv",
    index=False
)

mybl2_direction.to_csv(
    OUT /
    "Step33_MYBL2_MS3_PrimaryPatient_DirectionConsistency.csv",
    index=False
)

print("=" * 70)

In [ ]:
# Cell 53 — MYBL2 response topology + final report

import json

print("=" * 70)
print("Cell 53 — MYBL2 FINAL REPORT")
print("=" * 70)

mybl2_direct_targets = set(
    grn_sim.loc[
        grn_sim["TF"] == "MYBL2",
        "target"
    ].astype(str)
)

response_rows = []

for endpoint, genes in MYBL2_ENDPOINTS.items():

    idx_ep = [
        mybl2_score_genes.index(g)
        for g in genes
    ]

    gene_delta = (
        z_sim_mybl2[
            ms3_mask
        ][:, idx_ep]
        -
        z_base_mybl2[
            ms3_mask
        ][:, idx_ep]
    )

    med = np.median(
        gene_delta,
        axis=0
    )

    response_rows.append({
        "endpoint": endpoint,
        "simulatable_genes":
            len(genes),
        "genes_with_nonzero_median_delta":
            int(
                np.sum(
                    np.abs(med) > 1e-12
                )
            ),
        "direct_target_overlap_n":
            int(
                sum(
                    g in mybl2_direct_targets
                    for g in genes
                )
            ),
        "median_gene_delta":
            float(np.median(med)),
        "min_gene_delta":
            float(np.min(med)),
        "max_gene_delta":
            float(np.max(med)),
    })

mybl2_response = pd.DataFrame(
    response_rows
)

print("\nResponse topology:")
print(
    mybl2_response.to_string(
        index=False
    )
)

REPORT = {
    "status": "MYBL2_FORMAL_PERTURBATION_COMPLETE",

    "TF": "MYBL2",

    "n_propagation": 3,
    "use_randomized_GRN": False,
    "clip_delta_X": False,

    "MS3_cells": 720,

    "primary_replication_patients": [
        "HCC08",
        "HCC09",
        "HCC10"
    ],

    "overall_results":
        mybl2_overall.to_dict(
            orient="records"
        ),

    "patient_direction_consistency":
        mybl2_direction.to_dict(
            orient="records"
        ),

    "response_topology":
        mybl2_response.to_dict(
            orient="records"
        ),

    "claim_boundary":
        (
            "In silico counterfactual compatibility only; "
            "not causal validation or therapeutic efficacy."
        )
}

REPORT_PATH = (
    OUT /
    "Step33_MYBL2_FINAL_REPORT.json"
)

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        REPORT,
        f,
        indent=2,
        ensure_ascii=False
    )

mybl2_response.to_csv(
    OUT /
    "Step33_MYBL2_Endpoint_ResponseTopology.csv",
    index=False
)

print()
print("Saved:")
print(REPORT_PATH)

print("=" * 70)
print("MYBL2 FORMAL ANALYSIS: COMPLETE")
print("=" * 70)

In [ ]:
# Cell 54 — MYBL2 formal result lock

import json

print("=" * 70)
print("Cell 54 — MYBL2 FORMAL RESULT LOCK")
print("=" * 70)

# Correct naming: MYBL2 is not part of the current network-covered Core50,
# therefore no leave-target-out was actually required.

mybl2_overall_lock = mybl2_overall.copy()

mybl2_overall_lock["endpoint"] = (
    mybl2_overall_lock["endpoint"]
    .replace({
        "Core50_LTO_MYBL2":
            "Core50_NetworkCovered"
    })
)

mybl2_direction_lock = mybl2_direction.copy()

mybl2_direction_lock["endpoint"] = (
    mybl2_direction_lock["endpoint"]
    .replace({
        "Core50_LTO_MYBL2":
            "Core50_NetworkCovered"
    })
)

MYBL2_LOCK = {
    "status":
        "MYBL2_FORMAL_PERTURBATION_FROZEN",

    "TF":
        "MYBL2",

    "perturbation": {
        "n_propagation": 3,
        "use_randomized_GRN": False,
        "clip_delta_X": False,
    },

    "primary_population": {
        "state": "MS3",
        "cells": 720,
        "primary_replication_patients": [
            "HCC08",
            "HCC09",
            "HCC10"
        ],
        "sparse_sensitivity_patients": [
            "HCC04",
            "HCC07"
        ]
    },

    "core50_leave_target_out_required":
        False,

    "overall_results":
        mybl2_overall_lock.to_dict(
            orient="records"
        ),

    "patient_direction_consistency":
        mybl2_direction_lock.to_dict(
            orient="records"
        ),

    "interpretation_lock": (
        "MYBL2 virtual suppression predicts broad weakening "
        "of the network-reachable MS3 proliferative/mitotic "
        "transcriptional programs across HCC08, HCC09, and HCC10. "
        "No predicted restoration of hepatic differentiation or "
        "bile-acid metabolic programs was observed."
    ),

    "claim_boundary": (
        "In silico counterfactual prediction only; "
        "not causal validation, therapeutic efficacy, "
        "or lineage conversion."
    )
}

MYBL2_LOCK_PATH = (
    OUT /
    "Step33_MYBL2_FORMAL_FINAL_LOCK.json"
)

with open(
    MYBL2_LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        MYBL2_LOCK,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    MYBL2_LOCK,
    indent=2,
    ensure_ascii=False
))

print()
print("Saved:")
print(MYBL2_LOCK_PATH)

print("=" * 70)
print("MYBL2: FROZEN")
print("=" * 70)

In [ ]:
# Cell 55 — High-variable-gene gate audit for remaining formal TFs

import pandas as pd
import numpy as np

print("=" * 70)
print("Cell 55 — Remaining TF high-var gate audit")
print("=" * 70)

REMAINING_TFS = [
    "E2F8",
    "TFDP1",
    "E2F1"
]

assert {"TF", "target"}.issubset(grn_sim.columns)

rows = []

for tf in REMAINING_TFS:

    idx = oracle_final.adata.var_names.get_loc(tf)

    raw = oracle_final.adata.layers[
        "raw_count"
    ][:, idx]

    norm = oracle_final.adata.layers[
        "normalized_count"
    ][:, idx]

    imp = oracle_final.adata.layers[
        "imputed_count"
    ][:, idx]

    def vec(x):
        if hasattr(x, "toarray"):
            x = x.toarray()
        return np.asarray(x).ravel()

    raw = vec(raw)
    norm = vec(norm)
    imp = vec(imp)

    var_row = oracle_final.adata.var.loc[tf]

    frozen_target_n = int(
        grn_sim.loc[
            grn_sim["TF"] == tf,
            "target"
        ].nunique()
    )

    rows.append({
        "TF": tf,

        "in_all_regulatory":
            tf in oracle_final.all_regulatory_genes_in_TFdict,

        "in_active_regulatory":
            tf in oracle_final.active_regulatory_genes,

        "in_high_var_genes":
            tf in oracle_final.high_var_genes,

        "isin_top1000_var_mean_genes":
            bool(
                var_row[
                    "isin_top1000_var_mean_genes"
                ]
            ),

        "raw_detection_fraction":
            float(np.mean(raw > 0)),

        "normalized_mean":
            float(np.mean(norm)),

        "normalized_sd":
            float(np.std(norm)),

        "imputed_mean":
            float(np.mean(imp)),

        "imputed_sd":
            float(np.std(imp)),

        "imputed_min":
            float(np.min(imp)),

        "imputed_max":
            float(np.max(imp)),

        "frozen_target_n":
            frozen_target_n
    })

highvar_audit = pd.DataFrame(rows)

print(
    highvar_audit.to_string(
        index=False
    )
)

HIGHVAR_PATH = (
    OUT /
    "Step33_RemainingTF_HighVarGate_Audit.csv"
)

highvar_audit.to_csv(
    HIGHVAR_PATH,
    index=False
)

print()
print("Saved:")
print(HIGHVAR_PATH)

print("=" * 70)

In [ ]:
# Cell 56 — Inspect CellOracle high-var gate in simulation source

import inspect

print("=" * 70)
print("Cell 56 — CellOracle high-var gate source audit")
print("=" * 70)

# Private method used internally by simulate_shift
simulate_core = getattr(
    oracle_final,
    "_Oracle__simulate_shift"
)

src = inspect.getsource(simulate_core)
lines = src.splitlines()

keywords = [
    "high_var_genes",
    "active_regulatory_genes",
    "all_regulatory_genes_in_TFdict",
    "perturb_condition",
    "coef_matrix_per_cluster",
]

for i, line in enumerate(lines):

    if any(k in line for k in keywords):

        start = max(0, i - 4)
        end = min(len(lines), i + 6)

        print("\n" + "-" * 70)

        for j in range(start, end):
            print(f"{j+1:04d}: {lines[j]}")

print("\n" + "=" * 70)
print("Occurrences of high_var_genes in __simulate_shift:")
print(
    sum(
        "high_var_genes" in x
        for x in lines
    )
)

print("=" * 70)

In [ ]:
# Cell 57 — Search all high_var_genes usage in Oracle class

print("=" * 70)
print("Cell 57 — Oracle-wide high_var_genes usage")
print("=" * 70)

class_src = inspect.getsource(
    type(oracle_final)
)

class_lines = class_src.splitlines()

hits = []

for i, line in enumerate(class_lines):

    if "high_var_genes" in line:

        hits.append(i)

        print("\n" + "-" * 70)

        start = max(0, i - 4)
        end = min(len(class_lines), i + 6)

        for j in range(start, end):
            print(
                f"{j+1:05d}: {class_lines[j]}"
            )

print()
print(
    "Total high_var_genes occurrences:",
    len(hits)
)

print("=" * 70)

In [ ]:
# Cell 58 — High-var gate method lock

import json

print("=" * 70)
print("Cell 58 — CellOracle high-var gate METHOD LOCK")
print("=" * 70)

HIGHVAR_METHOD_LOCK = {
    "status": "HIGH_VAR_GATE_AUDITED",

    "remaining_formal_TFs": [
        "E2F8",
        "TFDP1",
        "E2F1"
    ],

    "all_in_base_GRN": True,
    "all_active_regulators": True,
    "all_high_var": False,

    "source_audit": {
        "high_var_occurrences_in_simulate_shift": 1,
        "high_var_role": "simulation QC metadata",
        "blocks_simulation": False,
        "used_in_propagation": False,
        "propagation_uses":
            "coef_matrix_per_cluster"
    },

    "compatibility_override_required": False,
    "manual_append_to_high_var_genes": False,
    "GRN_refit_required": False,
    "network_reconstruction": False,

    "interpretation": (
        "E2F8, TFDP1, and E2F1 are active regulators with fitted "
        "cluster-specific GRN coefficients. Their absence from the "
        "top-variable-gene QC set does not block or alter simulation "
        "in the installed CellOracle implementation."
    )
}

PATH = (
    OUT /
    "Step33_RemainingTF_HighVarGate_METHOD_LOCK.json"
)

with open(PATH, "w", encoding="utf-8") as f:
    json.dump(
        HIGHVAR_METHOD_LOCK,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(
    HIGHVAR_METHOD_LOCK,
    indent=2,
    ensure_ascii=False
))

print()
print("Saved:")
print(PATH)
print("=" * 70)

In [ ]:
# Cell 59 — E2F8 virtual knockout + technical QC

import numpy as np

print("=" * 70)
print("Cell 59 — E2F8 virtual knockout")
print("=" * 70)

KO_TF = "E2F8"

# Eligibility
assert KO_TF in oracle_final.all_regulatory_genes_in_TFdict
assert KO_TF in oracle_final.active_regulatory_genes

print(
    "in_high_var_genes:",
    KO_TF in oracle_final.high_var_genes,
    "(QC flag only; source-audited)"
)

# --------------------------------------------------
# Simulation
# --------------------------------------------------

oracle_final.simulate_shift(
    perturb_condition={
        KO_TF: 0.0
    },
    n_propagation=3,
    use_randomized_GRN=False,
    clip_delta_X=False
)

# --------------------------------------------------
# Technical manipulation check
# --------------------------------------------------

gene_idx = (
    oracle_final.adata.var_names
    .get_loc(KO_TF)
)

e2f8_base = get_gene_vector(
    "imputed_count",
    gene_idx
)

e2f8_sim = get_gene_vector(
    "simulated_count",
    gene_idx
)

e2f8_delta = (
    e2f8_sim - e2f8_base
)

print()
print(
    "Baseline median:",
    float(np.median(e2f8_base))
)

print(
    "Simulated median:",
    float(np.median(e2f8_sim))
)

print(
    "Delta median:",
    float(np.median(e2f8_delta))
)

print(
    "Simulated min/max:",
    float(np.min(e2f8_sim)),
    float(np.max(e2f8_sim))
)

print()
print("Running distribution-range QC...")

oracle_final.evaluate_simulated_gene_distribution_range()

print()
print("Distribution-range QC completed.")
print("=" * 70)

# Cell 60–63 — Unified Formal Scoring for Remaining Step33 TFs

The same frozen framework is reused for E2F8, TFDP1, and E2F1.

- Baseline: `imputed_count`
- Baseline-derived mean/SD only
- Current KO: `simulated_count`
- Automatic leave-target-out when the perturbed TF belongs to an endpoint
- Primary biological replication: HCC08, HCC09, HCC10
- HCC04/HCC07: sparse-cell sensitivity
- No GRN refitting or network reconstruction

In [ ]:
# Cell 60 — Unified Step33 perturbation scoring function

import numpy as np
import pandas as pd
import scipy.sparse as sp
import json

print("=" * 70)
print("Cell 60 — Build unified Step33 scoring function")
print("=" * 70)

PRIMARY_PATIENTS = ["HCC08", "HCC09", "HCC10"]

# Canonical network-covered frozen endpoints
BASE_ENDPOINTS = {
    "Core50_NetworkCovered":
        SIM_ENDPOINTS["Core50_Full"],

    "G2M_Independent":
        SIM_ENDPOINTS["G2M_Independent"],

    "Mitotic_Entry_Independent":
        SIM_ENDPOINTS["Mitotic_Entry_Independent"],

    "Spindle_Chromosome_Independent":
        SIM_ENDPOINTS["Spindle_Chromosome_Independent"],

    "Hepatic_Differentiation":
        SIM_ENDPOINTS["Hepatic_Differentiation"],

    "Bile_Acid_Metabolism":
        SIM_ENDPOINTS["Bile_Acid_Metabolism"],
}

EXPECTED_DIRECTION = {
    "Core50_NetworkCovered": "negative",
    "G2M_Independent": "negative",
    "Mitotic_Entry_Independent": "negative",
    "Spindle_Chromosome_Independent": "negative",
    "Hepatic_Differentiation": "positive",
    "Bile_Acid_Metabolism": "positive",
}


def _dense(x):
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x)


def score_current_step33_simulation(tf):

    print("\n" + "=" * 70)
    print(f"Formal scoring — {tf}")
    print("=" * 70)

    # --------------------------------------------------
    # TF-specific leave-target-out
    # --------------------------------------------------

    endpoints = {}
    endpoint_meta = []

    for endpoint, genes in BASE_ENDPOINTS.items():

        lto = tf in genes

        final_genes = [
            g for g in genes
            if g != tf
        ]

        endpoints[endpoint] = final_genes

        endpoint_meta.append({
            "endpoint": endpoint,
            "TF": tf,
            "network_covered_before_LTO": len(genes),
            "LTO_applied": bool(lto),
            "genes_after_LTO": len(final_genes),
        })

    endpoint_meta = pd.DataFrame(endpoint_meta)

    print("\nEndpoint design:")
    print(endpoint_meta.to_string(index=False))

    # --------------------------------------------------
    # Gene union
    # --------------------------------------------------

    score_genes = sorted(
        set(
            g
            for genes in endpoints.values()
            for g in genes
        )
    )

    gene_pos = {
        g: i for i, g in enumerate(score_genes)
    }

    var_idx = [
        oracle_final.adata.var_names.get_loc(g)
        for g in score_genes
    ]

    baseline = _dense(
        oracle_final.adata.layers[
            "imputed_count"
        ][:, var_idx]
    ).astype(np.float32)

    simulated = _dense(
        oracle_final.adata.layers[
            "simulated_count"
        ][:, var_idx]
    ).astype(np.float32)

    # --------------------------------------------------
    # Baseline-anchored normalization
    # --------------------------------------------------

    mu = baseline.mean(
        axis=0,
        dtype=np.float64
    )

    sigma = baseline.std(
        axis=0,
        ddof=0,
        dtype=np.float64
    )

    assert np.all(sigma > 1e-12)

    z_base = (
        baseline - mu
    ) / sigma

    z_sim = (
        simulated - mu
    ) / sigma

    # --------------------------------------------------
    # MS3 cells
    # --------------------------------------------------

    ms3_mask_local = (
        oracle_final.adata.obs[
            "final_malignant_state_id"
        ]
        .astype(str)
        .values
        ==
        "MS3"
    )

    assert int(ms3_mask_local.sum()) == 720

    scores = pd.DataFrame(
        index=oracle_final.adata.obs_names[
            ms3_mask_local
        ]
    )

    scores["patient"] = (
        oracle_final.adata.obs.loc[
            ms3_mask_local,
            "patient"
        ]
        .astype(str)
        .values
    )

    # --------------------------------------------------
    # Program scoring
    # --------------------------------------------------

    overall_rows = []
    patient_rows = []
    topology_rows = []

    direct_targets = set(
        grn_sim.loc[
            grn_sim["TF"] == tf,
            "target"
        ].astype(str)
    )

    for endpoint, genes in endpoints.items():

        ep_idx = [
            gene_pos[g]
            for g in genes
        ]

        base_score = (
            z_base[
                ms3_mask_local
            ][:, ep_idx]
            .mean(axis=1)
        )

        sim_score = (
            z_sim[
                ms3_mask_local
            ][:, ep_idx]
            .mean(axis=1)
        )

        delta = (
            sim_score
            -
            base_score
        )

        scores[
            f"{endpoint}__baseline"
        ] = base_score

        scores[
            f"{endpoint}__simulated"
        ] = sim_score

        scores[
            f"{endpoint}__delta"
        ] = delta

        overall_rows.append({
            "TF": tf,
            "endpoint": endpoint,
            "genes_after_LTO": len(genes),
            "LTO_applied":
                bool(tf in BASE_ENDPOINTS[endpoint]),

            "median_delta":
                float(np.median(delta)),

            "Q1_delta":
                float(np.quantile(delta, 0.25)),

            "Q3_delta":
                float(np.quantile(delta, 0.75)),

            "mean_delta":
                float(np.mean(delta)),

            "fraction_negative":
                float(np.mean(delta < 0)),

            "fraction_positive":
                float(np.mean(delta > 0)),
        })

        # patient summaries
        temp = pd.DataFrame({
            "patient": scores["patient"].values,
            "delta": delta
        })

        for patient, sub in temp.groupby("patient"):

            patient_rows.append({
                "TF": tf,
                "endpoint": endpoint,
                "patient": patient,
                "n_cells": len(sub),

                "median_delta":
                    float(np.median(sub["delta"])),

                "patient_role":
                    (
                        "primary_replication"
                        if patient in PRIMARY_PATIENTS
                        else
                        "sparse_sensitivity"
                    )
            })

        # gene-level response topology
        gene_delta = (
            z_sim[
                ms3_mask_local
            ][:, ep_idx]
            -
            z_base[
                ms3_mask_local
            ][:, ep_idx]
        )

        gene_med = np.median(
            gene_delta,
            axis=0
        )

        topology_rows.append({
            "TF": tf,
            "endpoint": endpoint,
            "genes_after_LTO": len(genes),

            "genes_with_nonzero_median_delta":
                int(
                    np.sum(
                        np.abs(gene_med) > 1e-12
                    )
                ),

            "direct_target_overlap_n":
                int(
                    sum(
                        g in direct_targets
                        for g in genes
                    )
                ),

            "median_gene_delta":
                float(np.median(gene_med)),

            "min_gene_delta":
                float(np.min(gene_med)),

            "max_gene_delta":
                float(np.max(gene_med)),
        })

    overall = pd.DataFrame(overall_rows)
    patient = pd.DataFrame(patient_rows)
    topology = pd.DataFrame(topology_rows)

    # --------------------------------------------------
    # Patient direction consistency
    # --------------------------------------------------

    direction_rows = []

    for endpoint in endpoints:

        sub = patient.loc[
            (
                patient["endpoint"] == endpoint
            )
            &
            (
                patient["patient"]
                .isin(PRIMARY_PATIENTS)
            )
        ]

        expected = EXPECTED_DIRECTION[
            endpoint
        ]

        if expected == "negative":
            ok = sub["median_delta"] < 0
        else:
            ok = sub["median_delta"] > 0

        direction_rows.append({
            "TF": tf,
            "endpoint": endpoint,
            "expected_direction": expected,
            "primary_patients_n": len(sub),
            "concordant_patients":
                int(ok.sum()),
            "direction_consistency":
                f"{int(ok.sum())}/{len(sub)}"
        })

    direction = pd.DataFrame(
        direction_rows
    )

    # --------------------------------------------------
    # Save
    # --------------------------------------------------

    prefix = f"Step33_{tf}"

    scores.to_csv(
        OUT /
        f"{prefix}_MS3_CellLevel_BaselineAnchoredScores.csv"
    )

    overall.to_csv(
        OUT /
        f"{prefix}_MS3_Overall_ProgramDelta.csv",
        index=False
    )

    patient.to_csv(
        OUT /
        f"{prefix}_MS3_PatientLevel_ProgramDelta.csv",
        index=False
    )

    direction.to_csv(
        OUT /
        f"{prefix}_MS3_PrimaryPatient_DirectionConsistency.csv",
        index=False
    )

    topology.to_csv(
        OUT /
        f"{prefix}_Endpoint_ResponseTopology.csv",
        index=False
    )

    endpoint_meta.to_csv(
        OUT /
        f"{prefix}_Endpoint_LTO_Audit.csv",
        index=False
    )

    report = {
        "status":
            f"{tf}_FORMAL_PERTURBATION_COMPLETE",

        "TF": tf,

        "n_propagation": 3,
        "use_randomized_GRN": False,
        "clip_delta_X": False,

        "MS3_cells": 720,

        "primary_replication_patients":
            PRIMARY_PATIENTS,

        "overall":
            overall.to_dict(
                orient="records"
            ),

        "direction_consistency":
            direction.to_dict(
                orient="records"
            ),

        "response_topology":
            topology.to_dict(
                orient="records"
            ),

        "claim_boundary": (
            "In silico counterfactual prediction only; "
            "not causal validation, therapeutic efficacy, "
            "or lineage conversion."
        )
    }

    with open(
        OUT /
        f"{prefix}_FINAL_REPORT.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            report,
            f,
            indent=2,
            ensure_ascii=False
        )

    print("\nOverall:")
    print(overall.to_string(index=False))

    print("\nPrimary patient direction consistency:")
    print(direction.to_string(index=False))

    print("\nResponse topology:")
    print(topology.to_string(index=False))

    print("\nSaved all formal outputs.")

    return {
        "overall": overall,
        "patient": patient,
        "direction": direction,
        "topology": topology,
        "scores": scores,
        "endpoint_meta": endpoint_meta,
    }


print("Unified scoring function ready.")
print("=" * 70)

In [ ]:
# Cell 61 — Score CURRENT E2F8 simulation

E2F8_RESULT = (
    score_current_step33_simulation(
        "E2F8"
    )
)

In [ ]:
# Cell 62 — TFDP1 formal simulation + scoring

print("=" * 70)
print("Cell 62 — TFDP1 formal simulation")
print("=" * 70)

KO_TF = "TFDP1"

assert KO_TF in oracle_final.all_regulatory_genes_in_TFdict
assert KO_TF in oracle_final.active_regulatory_genes

oracle_final.simulate_shift(
    perturb_condition={
        KO_TF: 0.0
    },
    n_propagation=3,
    use_randomized_GRN=False,
    clip_delta_X=False
)

idx = oracle_final.adata.var_names.get_loc(
    KO_TF
)

tf_base = get_gene_vector(
    "imputed_count",
    idx
)

tf_sim = get_gene_vector(
    "simulated_count",
    idx
)

print(
    "Baseline median:",
    float(np.median(tf_base))
)

print(
    "Simulated median:",
    float(np.median(tf_sim))
)

print(
    "Simulated min/max:",
    float(np.min(tf_sim)),
    float(np.max(tf_sim))
)

oracle_final.evaluate_simulated_gene_distribution_range()

print("Distribution-range QC completed.")

TFDP1_RESULT = (
    score_current_step33_simulation(
        "TFDP1"
    )
)

In [ ]:
# Cell 63 — E2F1 formal simulation + scoring

print("=" * 70)
print("Cell 63 — E2F1 formal simulation")
print("=" * 70)

KO_TF = "E2F1"

assert KO_TF in oracle_final.all_regulatory_genes_in_TFdict
assert KO_TF in oracle_final.active_regulatory_genes

oracle_final.simulate_shift(
    perturb_condition={
        KO_TF: 0.0
    },
    n_propagation=3,
    use_randomized_GRN=False,
    clip_delta_X=False
)

idx = oracle_final.adata.var_names.get_loc(
    KO_TF
)

tf_base = get_gene_vector(
    "imputed_count",
    idx
)

tf_sim = get_gene_vector(
    "simulated_count",
    idx
)

print(
    "Baseline median:",
    float(np.median(tf_base))
)

print(
    "Simulated median:",
    float(np.median(tf_sim))
)

print(
    "Simulated min/max:",
    float(np.min(tf_sim)),
    float(np.max(tf_sim))
)

oracle_final.evaluate_simulated_gene_distribution_range()

print("Distribution-range QC completed.")

E2F1_RESULT = (
    score_current_step33_simulation(
        "E2F1"
    )
)

# Cell 64–66 — Five-TF Integration and Step33 Final Lock

Integrate the five frozen formal perturbations:

- FOXM1
- MYBL2
- E2F8
- TFDP1
- E2F1

Primary endpoints:
- Network-covered MS3 Core50
- G2M Independent
- Mitotic Entry Independent
- Spindle/Chromosome Independent

FOXM1 uses leave-target-out for Core50.
No additional GRN fitting, perturbation, feature selection, or endpoint redefinition is performed.

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# Cell 64 — Five-TF integrated summary

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 70)
print("Cell 64 — Five-TF integrated summary")
print("=" * 70)

OUT = Path(
    _hcc_path('LIHC_scRNA_Step33_Virtual_Knockout')
)

TF_ORDER = [
    "FOXM1",
    "MYBL2",
    "E2F8",
    "TFDP1",
    "E2F1",
]

PRIMARY_ENDPOINTS = [
    "Core50_NetworkCovered",
    "G2M_Independent",
    "Mitotic_Entry_Independent",
    "Spindle_Chromosome_Independent",
]

SECONDARY_ENDPOINTS = [
    "Hepatic_Differentiation",
    "Bile_Acid_Metabolism",
]

overall_files = {
    "FOXM1":
        OUT / "Step33_FOXM1_MS3_Overall_ProgramDelta.csv",

    "MYBL2":
        OUT / "Step33_MYBL2_MS3_Overall_ProgramDelta.csv",

    "E2F8":
        OUT / "Step33_E2F8_MS3_Overall_ProgramDelta.csv",

    "TFDP1":
        OUT / "Step33_TFDP1_MS3_Overall_ProgramDelta.csv",

    "E2F1":
        OUT / "Step33_E2F1_MS3_Overall_ProgramDelta.csv",
}

direction_files = {
    "FOXM1":
        OUT / "Step33_FOXM1_MS3_PrimaryPatient_DirectionConsistency.csv",

    "MYBL2":
        OUT / "Step33_MYBL2_MS3_PrimaryPatient_DirectionConsistency.csv",

    "E2F8":
        OUT / "Step33_E2F8_MS3_PrimaryPatient_DirectionConsistency.csv",

    "TFDP1":
        OUT / "Step33_TFDP1_MS3_PrimaryPatient_DirectionConsistency.csv",

    "E2F1":
        OUT / "Step33_E2F1_MS3_PrimaryPatient_DirectionConsistency.csv",
}


# --------------------------------------------------
# Load + normalize overall results
# --------------------------------------------------

overall_frames = []

for tf in TF_ORDER:

    df = pd.read_csv(
        overall_files[tf]
    )

    if "TF" not in df.columns:
        df["TF"] = tf

    # preserve original name
    df["original_endpoint"] = df["endpoint"]

    # Harmonize historical Core50 names
    df["endpoint"] = (
        df["endpoint"]
        .replace({
            "Core50_LTO_FOXM1":
                "Core50_NetworkCovered",

            "Core50_LTO_MYBL2":
                "Core50_NetworkCovered",

            "Core50_Full":
                "Core50_NetworkCovered",
        })
    )

    if "genes_after_LTO" not in df.columns:

        if "simulatable_genes" in df.columns:
            df["genes_after_LTO"] = (
                df["simulatable_genes"]
            )

    if "LTO_applied" not in df.columns:
        df["LTO_applied"] = False

    # FOXM1 Core50 is the only formal Core50 LTO case
    if tf == "FOXM1":

        mask = (
            df["original_endpoint"]
            ==
            "Core50_LTO_FOXM1"
        )

        df.loc[
            mask,
            "LTO_applied"
        ] = True

    overall_frames.append(df)

integrated_overall = pd.concat(
    overall_frames,
    ignore_index=True
)


# --------------------------------------------------
# Load + normalize patient-direction results
# --------------------------------------------------

direction_frames = []

for tf in TF_ORDER:

    df = pd.read_csv(
        direction_files[tf]
    )

    if "TF" not in df.columns:
        df["TF"] = tf

    df["endpoint"] = (
        df["endpoint"]
        .replace({
            "Core50_LTO_FOXM1":
                "Core50_NetworkCovered",

            "Core50_LTO_MYBL2":
                "Core50_NetworkCovered",

            "Core50_Full":
                "Core50_NetworkCovered",
        })
    )

    direction_frames.append(df)

integrated_direction = pd.concat(
    direction_frames,
    ignore_index=True
)


# --------------------------------------------------
# Merge
# --------------------------------------------------

integrated = integrated_overall.merge(
    integrated_direction[
        [
            "TF",
            "endpoint",
            "direction_consistency",
            "concordant_patients",
            "primary_patients_n",
        ]
    ],
    on=[
        "TF",
        "endpoint"
    ],
    how="left"
)


# --------------------------------------------------
# Primary integrated table
# --------------------------------------------------

primary_integrated = (
    integrated.loc[
        integrated["endpoint"]
        .isin(PRIMARY_ENDPOINTS)
    ]
    .copy()
)

primary_integrated["TF"] = pd.Categorical(
    primary_integrated["TF"],
    categories=TF_ORDER,
    ordered=True
)

primary_integrated["endpoint"] = pd.Categorical(
    primary_integrated["endpoint"],
    categories=PRIMARY_ENDPOINTS,
    ordered=True
)

primary_integrated = (
    primary_integrated
    .sort_values(
        ["TF", "endpoint"]
    )
    .reset_index(drop=True)
)


print("\nPrimary integrated results:")

print(
    primary_integrated[
        [
            "TF",
            "endpoint",
            "genes_after_LTO",
            "LTO_applied",
            "median_delta",
            "Q1_delta",
            "Q3_delta",
            "fraction_negative",
            "direction_consistency",
        ]
    ].to_string(index=False)
)


# --------------------------------------------------
# Cross-TF convergence
# --------------------------------------------------

convergence_rows = []

for endpoint in PRIMARY_ENDPOINTS:

    sub = primary_integrated.loc[
        primary_integrated["endpoint"]
        == endpoint
    ]

    convergence_rows.append({
        "endpoint": endpoint,
        "formal_TFs_n": len(sub),

        "TFs_with_negative_median":
            int(
                (sub["median_delta"] < 0)
                .sum()
            ),

        "TFs_with_3of3_patient_consistency":
            int(
                (
                    sub[
                        "direction_consistency"
                    ]
                    ==
                    "3/3"
                ).sum()
            ),

        "median_across_TFs":
            float(
                np.median(
                    sub["median_delta"]
                )
            ),

        "min_delta":
            float(
                sub["median_delta"].min()
            ),

        "max_delta":
            float(
                sub["median_delta"].max()
            ),
    })

convergence_df = pd.DataFrame(
    convergence_rows
)

print()
print("Cross-TF convergence:")

print(
    convergence_df.to_string(
        index=False
    )
)


# --------------------------------------------------
# Secondary directional audit
# --------------------------------------------------

secondary_integrated = (
    integrated.loc[
        integrated["endpoint"]
        .isin(SECONDARY_ENDPOINTS)
    ]
    .copy()
)

print()
print("Secondary directional endpoints:")

print(
    secondary_integrated[
        [
            "TF",
            "endpoint",
            "median_delta",
            "direction_consistency",
        ]
    ].to_string(index=False)
)


# --------------------------------------------------
# Save
# --------------------------------------------------

integrated.to_csv(
    OUT /
    "Step33_5TF_Integrated_AllEndpoints.csv",
    index=False
)

primary_integrated.to_csv(
    OUT /
    "Step33_5TF_Integrated_PrimaryEndpoints.csv",
    index=False
)

convergence_df.to_csv(
    OUT /
    "Step33_5TF_PrimaryEndpoint_Convergence.csv",
    index=False
)

secondary_integrated.to_csv(
    OUT /
    "Step33_5TF_SecondaryDirectionalEndpoints.csv",
    index=False
)

print()
print("Integrated tables saved.")
print("=" * 70)

In [ ]:
# Cell 64B — Fix five-TF integration naming collision

import pandas as pd
import numpy as np

print("=" * 70)
print("Cell 64B — Correct five-TF integration")
print("=" * 70)

TF_ORDER = [
    "FOXM1",
    "MYBL2",
    "E2F8",
    "TFDP1",
    "E2F1",
]

PRIMARY_ENDPOINTS = [
    "Core50_NetworkCovered",
    "G2M_Independent",
    "Mitotic_Entry_Independent",
    "Spindle_Chromosome_Independent",
]

SECONDARY_ENDPOINTS = [
    "Hepatic_Differentiation",
    "Bile_Acid_Metabolism",
]

# --------------------------------------------------
# Load overall tables with CORRECT endpoint mapping
# --------------------------------------------------

overall_frames = []

for tf in TF_ORDER:

    df = pd.read_csv(
        overall_files[tf]
    )

    if "TF" not in df.columns:
        df["TF"] = tf

    df["original_endpoint"] = df["endpoint"]

    # IMPORTANT:
    # FOXM1 LTO = formal primary Core50
    # FOXM1 Full = descriptive only, keep separate
    mapping = {
        "Core50_LTO_FOXM1":
            "Core50_NetworkCovered",

        "Core50_LTO_MYBL2":
            "Core50_NetworkCovered",

        "Core50_Full":
            "Core50_NetworkCovered_Full_Descriptive",
    }

    df["endpoint"] = (
        df["endpoint"]
        .replace(mapping)
    )

    if "genes_after_LTO" not in df.columns:

        if "simulatable_genes" in df.columns:
            df["genes_after_LTO"] = (
                df["simulatable_genes"]
            )

    if "LTO_applied" not in df.columns:
        df["LTO_applied"] = False

    # Only FOXM1 formal Core50 endpoint actually uses LTO
    mask_foxm1 = (
        (df["TF"] == "FOXM1")
        &
        (
            df["original_endpoint"]
            == "Core50_LTO_FOXM1"
        )
    )

    df.loc[
        mask_foxm1,
        "LTO_applied"
    ] = True

    overall_frames.append(df)

integrated_overall = pd.concat(
    overall_frames,
    ignore_index=True
)

# --------------------------------------------------
# Direction tables
# --------------------------------------------------

direction_frames = []

for tf in TF_ORDER:

    df = pd.read_csv(
        direction_files[tf]
    )

    if "TF" not in df.columns:
        df["TF"] = tf

    df["original_endpoint"] = df["endpoint"]

    df["endpoint"] = (
        df["endpoint"]
        .replace({
            "Core50_LTO_FOXM1":
                "Core50_NetworkCovered",

            "Core50_LTO_MYBL2":
                "Core50_NetworkCovered",

            "Core50_Full":
                "Core50_NetworkCovered_Full_Descriptive",
        })
    )

    direction_frames.append(df)

integrated_direction = pd.concat(
    direction_frames,
    ignore_index=True
)

# --------------------------------------------------
# Merge
# --------------------------------------------------

integrated = integrated_overall.merge(
    integrated_direction[
        [
            "TF",
            "endpoint",
            "direction_consistency",
            "concordant_patients",
            "primary_patients_n",
        ]
    ],
    on=[
        "TF",
        "endpoint"
    ],
    how="left"
)

# --------------------------------------------------
# Primary table
# --------------------------------------------------

primary_integrated = (
    integrated.loc[
        integrated["endpoint"]
        .isin(PRIMARY_ENDPOINTS)
    ]
    .copy()
)

# Critical uniqueness check
dup = (
    primary_integrated
    .duplicated(
        subset=[
            "TF",
            "endpoint"
        ],
        keep=False
    )
)

print(
    "Duplicate TF-endpoint rows:",
    int(dup.sum())
)

if dup.any():

    print(
        primary_integrated.loc[
            dup,
            [
                "TF",
                "endpoint",
                "original_endpoint"
            ]
        ].to_string(index=False)
    )

assert not dup.any(), (
    "Primary integration still contains duplicated "
    "TF-endpoint combinations."
)

primary_integrated["TF"] = pd.Categorical(
    primary_integrated["TF"],
    categories=TF_ORDER,
    ordered=True
)

primary_integrated["endpoint"] = pd.Categorical(
    primary_integrated["endpoint"],
    categories=PRIMARY_ENDPOINTS,
    ordered=True
)

primary_integrated = (
    primary_integrated
    .sort_values(
        [
            "TF",
            "endpoint"
        ]
    )
    .reset_index(drop=True)
)

print()
print("Primary rows:", len(primary_integrated))

assert len(primary_integrated) == 20  # 5 TF × 4 endpoints

print()
print(
    primary_integrated[
        [
            "TF",
            "endpoint",
            "genes_after_LTO",
            "LTO_applied",
            "median_delta",
            "direction_consistency",
        ]
    ].to_string(index=False)
)

# --------------------------------------------------
# Convergence
# --------------------------------------------------

convergence_rows = []

for endpoint in PRIMARY_ENDPOINTS:

    sub = primary_integrated.loc[
        primary_integrated["endpoint"]
        == endpoint
    ]

    convergence_rows.append({
        "endpoint":
            endpoint,

        "formal_TFs_n":
            len(sub),

        "TFs_with_negative_median":
            int(
                (
                    sub["median_delta"]
                    < 0
                ).sum()
            ),

        "TFs_with_3of3_patient_consistency":
            int(
                (
                    sub["direction_consistency"]
                    == "3/3"
                ).sum()
            ),

        "median_across_TFs":
            float(
                np.median(
                    sub["median_delta"]
                )
            ),

        "min_delta":
            float(
                sub["median_delta"]
                .min()
            ),

        "max_delta":
            float(
                sub["median_delta"]
                .max()
            ),
    })

convergence_df = pd.DataFrame(
    convergence_rows
)

print()
print("Cross-TF convergence:")
print(
    convergence_df.to_string(
        index=False
    )
)

# --------------------------------------------------
# Secondary directional endpoints
# --------------------------------------------------

secondary_integrated = (
    integrated.loc[
        integrated["endpoint"]
        .isin(SECONDARY_ENDPOINTS)
    ]
    .copy()
)

# --------------------------------------------------
# Descriptive Core50 Full kept separately
# --------------------------------------------------

descriptive_core50 = (
    integrated.loc[
        integrated["endpoint"]
        ==
        "Core50_NetworkCovered_Full_Descriptive"
    ]
    .copy()
)

print()
print(
    "Descriptive full-Core50 rows:",
    len(descriptive_core50)
)

# --------------------------------------------------
# Save corrected integrated outputs
# --------------------------------------------------

integrated.to_csv(
    OUT /
    "Step33_5TF_Integrated_AllEndpoints.csv",
    index=False
)

primary_integrated.to_csv(
    OUT /
    "Step33_5TF_Integrated_PrimaryEndpoints.csv",
    index=False
)

convergence_df.to_csv(
    OUT /
    "Step33_5TF_PrimaryEndpoint_Convergence.csv",
    index=False
)

secondary_integrated.to_csv(
    OUT /
    "Step33_5TF_SecondaryDirectionalEndpoints.csv",
    index=False
)

descriptive_core50.to_csv(
    OUT /
    "Step33_Core50_Full_Descriptive_Only.csv",
    index=False
)

print()
print("=" * 70)
print("Corrected five-TF integration: PASS")
print("=" * 70)

In [ ]:
# Cell 65 — Step33 five-TF primary result heatmap

import matplotlib.pyplot as plt
import numpy as np

print("=" * 70)
print("Cell 65 — Step33 primary result figure")
print("=" * 70)

FIG_DIR = OUT / "Figures"
FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

plot_df = (
    primary_integrated
    .pivot(
        index="TF",
        columns="endpoint",
        values="median_delta"
    )
    .reindex(
        index=TF_ORDER,
        columns=PRIMARY_ENDPOINTS
    )
)

direction_df = (
    primary_integrated
    .pivot(
        index="TF",
        columns="endpoint",
        values="direction_consistency"
    )
    .reindex(
        index=TF_ORDER,
        columns=PRIMARY_ENDPOINTS
    )
)

display_names = [
    "MS3 Core50",
    "G2M",
    "Mitotic Entry",
    "Spindle / Chromosome",
]

fig, ax = plt.subplots(
    figsize=(9.0, 5.2)
)

im = ax.imshow(
    plot_df.values,
    aspect="auto"
)

ax.set_xticks(
    np.arange(
        len(PRIMARY_ENDPOINTS)
    )
)

ax.set_xticklabels(
    display_names,
    rotation=25,
    ha="right"
)

ax.set_yticks(
    np.arange(
        len(TF_ORDER)
    )
)

ax.set_yticklabels(
    TF_ORDER
)

ax.set_title(
    "Predicted weakening of MS3 proliferative/mitotic programs\n"
    "after virtual regulator suppression"
)

ax.set_xlabel(
    "Frozen MS3 program"
)

ax.set_ylabel(
    "Virtual knockout"
)


# annotate
for i, tf in enumerate(TF_ORDER):

    for j, endpoint in enumerate(
        PRIMARY_ENDPOINTS
    ):

        value = plot_df.loc[
            tf,
            endpoint
        ]

        consistency = direction_df.loc[
            tf,
            endpoint
        ]

        ax.text(
            j,
            i,
            f"{value:.3f}\n{consistency}",
            ha="center",
            va="center",
            fontsize=8
        )


cbar = fig.colorbar(
    im,
    ax=ax
)

cbar.set_label(
    "Median ΔProgram\n(KO − baseline)"
)

fig.tight_layout()

PNG_PATH = (
    FIG_DIR /
    "Step33_FiveTF_PrimaryProgramDelta_Heatmap.png"
)

PDF_PATH = (
    FIG_DIR /
    "Step33_FiveTF_PrimaryProgramDelta_Heatmap.pdf"
)

fig.savefig(
    PNG_PATH,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    PDF_PATH,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Saved:")
print(PNG_PATH)
print(PDF_PATH)

print("=" * 70)

In [ ]:
# Cell 66 — Step33 FINAL LOCK

import json
from datetime import datetime

print("=" * 70)
print("Cell 66 — STEP33 FINAL LOCK")
print("=" * 70)

# --------------------------------------------------
# Formal convergence checks
# --------------------------------------------------

all_primary_negative = bool(
    (
        primary_integrated[
            "median_delta"
        ]
        < 0
    ).all()
)

all_primary_3of3 = bool(
    (
        primary_integrated[
            "direction_consistency"
        ]
        ==
        "3/3"
    ).all()
)

secondary_all_zero = bool(
    np.allclose(
        secondary_integrated[
            "median_delta"
        ].values,
        0.0
    )
)

assert all_primary_negative
assert all_primary_3of3
assert secondary_all_zero


STEP33_FINAL_LOCK = {
    "status":
        "STEP33_FROZEN",

    "analysis":
        "Virtual Gene Knockout",

    "formal_TFs": [
        "FOXM1",
        "MYBL2",
        "E2F8",
        "TFDP1",
        "E2F1",
    ],

    "BRCA1_status":
        (
            "Frozen mechanistic candidate; "
            "not used as formal TF-specific knockout."
        ),

    "input": {
        "cells": 12057,
        "fitting_genes": 10776,
        "primary_state": "MS3",
        "MS3_cells": 720,
    },

    "network": {
        "source":
            "Step29 motif-filtered regulon topology",

        "original_edges":
            32530,

        "self_loops_removed":
            140,

        "simulation_edges":
            32390,

        "network_reconstruction":
            False,

        "TF_reselection":
            False,
    },

    "celloracle": {
        "ridge_alpha": 10,

        "PCA_dims": 13,

        "KNN_k": 301,

        "balanced_KNN": True,

        "b_sight": 2408,

        "b_maxl": 1204,

        "n_propagation": 3,

        "use_randomized_GRN":
            False,

        "clip_delta_X":
            False,
    },

    "scoring": {
        "method":
            "Baseline-Anchored CellOracle Perturbation Score",

        "baseline":
            "imputed_count",

        "perturbed":
            "simulated_count",

        "normalization": (
            "Gene mean and SD estimated once from "
            "baseline malignant cells; identical "
            "baseline parameters applied to KO."
        ),

        "delta_definition":
            "KO ProgramScore - baseline ProgramScore",

        "primary_endpoints": [
            "Core50_NetworkCovered",
            "G2M_Independent",
            "Mitotic_Entry_Independent",
            "Spindle_Chromosome_Independent",
        ],

        "secondary_directional_endpoints": [
            "Hepatic_Differentiation",
            "Bile_Acid_Metabolism",
        ],

        "FOXM1_Core50_LTO":
            True,
    },

    "biological_replication": {
        "primary_patients": [
            "HCC08",
            "HCC09",
            "HCC10",
        ],

        "sparse_sensitivity_patients": [
            "HCC04",
            "HCC07",
        ],

        "primary_unit":
            "patient",

        "cell_level_statistics":
            "descriptive only",
    },

    "convergence": {
        "formal_TFs_with_negative_primary_programs":
            "5/5",

        "formal_TFs_with_3of3_patient_direction_consistency":
            "5/5",

        "hepatic_differentiation_restoration":
            "not predicted",

        "bile_acid_metabolism_restoration":
            "not predicted",
    },

    "primary_results":
        primary_integrated[
            [
                "TF",
                "endpoint",
                "genes_after_LTO",
                "LTO_applied",
                "median_delta",
                "Q1_delta",
                "Q3_delta",
                "fraction_negative",
                "direction_consistency",
            ]
        ].to_dict(
            orient="records"
        ),

    "interpretation_lock": (
        "Across five frozen MS3-associated regulators, "
        "virtual suppression consistently predicted weakening "
        "of network-reachable proliferative/mitotic MS3 programs "
        "across HCC08, HCC09, and HCC10. "
        "The magnitude and network reach varied among regulators, "
        "but the directional convergence was preserved. "
        "No virtual perturbation predicted restoration of hepatic "
        "differentiation or bile-acid metabolic programs."
    ),

    "claim_boundary": (
        "These results represent in silico counterfactual "
        "compatibility within the frozen CellOracle GRN. "
        "They do not establish causal regulation, therapeutic "
        "efficacy, lineage conversion, or experimentally validated "
        "gene essentiality."
    ),

    "next_step":
        "Step34 — Virtual State Transition",
}


FINAL_LOCK_PATH = (
    OUT /
    "Step33_FINAL_LOCK.json"
)

with open(
    FINAL_LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        STEP33_FINAL_LOCK,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------
# Article-centered summary
# --------------------------------------------------

article_summary = pd.DataFrame([
    {
        "Step":
            "Step33",

        "Analysis":
            "Virtual Gene Knockout",

        "Formal_TFs":
            "FOXM1;MYBL2;E2F8;TFDP1;E2F1",

        "Primary_result":
            (
                "All five regulator suppressions predicted "
                "weakening of all four primary MS3 "
                "proliferative/mitotic programs."
            ),

        "Patient_consistency":
            (
                "All 5 TFs showed 3/3 concordant direction "
                "across HCC08/HCC09/HCC10 for all primary endpoints."
            ),

        "Differentiation_result":
            (
                "No predicted restoration of hepatic differentiation "
                "or bile-acid metabolism."
            ),

        "Claim_boundary":
            (
                "In silico counterfactual compatibility only; "
                "not causal or therapeutic validation."
            ),

        "Status":
            "FROZEN",
    }
])

ARTICLE_PATH = (
    OUT /
    "Step33_FINAL_ArticleCenteredSummary.csv"
)

article_summary.to_csv(
    ARTICLE_PATH,
    index=False
)


print("Five formal TFs: 5")
print("Primary endpoints: 4")
print(
    "All TF × endpoint median deltas negative:",
    all_primary_negative
)
print(
    "All primary patient consistency = 3/3:",
    all_primary_3of3
)
print(
    "Secondary hepatic/bile median deltas all zero:",
    secondary_all_zero
)

print()
print("Final lock:")
print(FINAL_LOCK_PATH)

print()
print("Article summary:")
print(ARTICLE_PATH)

print()
print("=" * 70)
print("STEP33 STATUS: FROZEN")
print("NEXT: STEP34 — VIRTUAL STATE TRANSITION")
print("=" * 70)

# Cell 67–73 — Step33 Publication Package and Final Archive

Step33 biological analyses are frozen.

This module performs publication packaging only:

- no new perturbation
- no GRN refitting
- no network reconstruction
- no feature or TF reselection
- no endpoint redefinition

Outputs:
- publication-ready main figures
- supplementary figures
- main tables
- supplementary tables
- figure/table legends
- Methods/Results claim lock
- reproducibility manifest
- final package lock

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# Cell 67 — Step33 publication package setup

from pathlib import Path
import pandas as pd
import numpy as np
import json
import shutil
import hashlib
import matplotlib.pyplot as plt

print("=" * 70)
print("Cell 67 — Step33 publication package setup")
print("=" * 70)

ROOT = Path(
    _hcc_path('LIHC_scRNA_Step33_Virtual_Knockout')
)

PKG = ROOT / "Publication_Package"

FIG_MAIN = PKG / "Figures" / "Main"
FIG_SUPP = PKG / "Figures" / "Supplementary"

TAB_MAIN = PKG / "Tables" / "Main"
TAB_SUPP = PKG / "Tables" / "Supplementary"

REPORT_DIR = PKG / "Reports"

for p in [
    FIG_MAIN,
    FIG_SUPP,
    TAB_MAIN,
    TAB_SUPP,
    REPORT_DIR
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )


TF_ORDER = [
    "FOXM1",
    "MYBL2",
    "E2F8",
    "TFDP1",
    "E2F1"
]

PRIMARY_ENDPOINTS = [
    "Core50_NetworkCovered",
    "G2M_Independent",
    "Mitotic_Entry_Independent",
    "Spindle_Chromosome_Independent"
]

SECONDARY_ENDPOINTS = [
    "Hepatic_Differentiation",
    "Bile_Acid_Metabolism"
]

ENDPOINT_DISPLAY = {
    "Core50_NetworkCovered":
        "MS3 Core50",

    "G2M_Independent":
        "G2M",

    "Mitotic_Entry_Independent":
        "Mitotic Entry",

    "Spindle_Chromosome_Independent":
        "Spindle / Chromosome",

    "Hepatic_Differentiation":
        "Hepatic Differentiation",

    "Bile_Acid_Metabolism":
        "Bile Acid Metabolism",
}


# --------------------------------------------------
# Verify frozen Step33
# --------------------------------------------------

FINAL_LOCK_PATH = (
    ROOT /
    "Step33_FINAL_LOCK.json"
)

assert FINAL_LOCK_PATH.exists()

with open(
    FINAL_LOCK_PATH,
    "r",
    encoding="utf-8"
) as f:
    step33_lock = json.load(f)

assert (
    step33_lock["status"]
    ==
    "STEP33_FROZEN"
)


# --------------------------------------------------
# Load integrated frozen tables
# --------------------------------------------------

primary = pd.read_csv(
    ROOT /
    "Step33_5TF_Integrated_PrimaryEndpoints.csv"
)

convergence = pd.read_csv(
    ROOT /
    "Step33_5TF_PrimaryEndpoint_Convergence.csv"
)

secondary = pd.read_csv(
    ROOT /
    "Step33_5TF_SecondaryDirectionalEndpoints.csv"
)

coverage = pd.read_csv(
    ROOT /
    "Step33_FROZEN_ENDPOINT_COVERAGE.csv"
)

projection = pd.read_csv(
    ROOT /
    "Step33_FROZEN_ENDPOINT_CELLORACLE_PROJECTION.csv"
)

eligibility = pd.read_csv(
    ROOT /
    "Step33_FORMAL_TF_ELIGIBILITY.csv"
)

highvar_audit = pd.read_csv(
    ROOT /
    "Step33_RemainingTF_HighVarGate_Audit.csv"
)


# --------------------------------------------------
# Core integrity checks
# --------------------------------------------------

assert len(primary) == 20
assert len(convergence) == 4

assert (
    primary["median_delta"] < 0
).all()

assert (
    primary["direction_consistency"]
    ==
    "3/3"
).all()

print("Primary rows:", len(primary))
print("Convergence rows:", len(convergence))

print()
print("Publication package root:")
print(PKG)

print("=" * 70)
print("Publication package setup: PASS")

In [ ]:
# Cell 68 — Build publication tables

print("=" * 70)
print("Cell 68 — Build publication tables")
print("=" * 70)


# ==================================================
# MAIN TABLE 1
# Five-TF primary results
# ==================================================

main_table1_cols = [
    "TF",
    "endpoint",
    "genes_after_LTO",
    "LTO_applied",
    "median_delta",
    "Q1_delta",
    "Q3_delta",
    "mean_delta",
    "fraction_negative",
    "direction_consistency"
]

main_table1 = (
    primary[
        [
            c for c in main_table1_cols
            if c in primary.columns
        ]
    ]
    .copy()
)

main_table1["endpoint"] = (
    main_table1["endpoint"]
    .map(ENDPOINT_DISPLAY)
)

main_table1.to_csv(
    TAB_MAIN /
    "Table1_Step33_FiveTF_PrimaryPerturbationResults.csv",
    index=False
)


# ==================================================
# MAIN TABLE 2
# Cross-TF convergence
# ==================================================

main_table2 = convergence.copy()

main_table2["endpoint"] = (
    main_table2["endpoint"]
    .map(ENDPOINT_DISPLAY)
)

main_table2.to_csv(
    TAB_MAIN /
    "Table2_Step33_CrossTF_Convergence.csv",
    index=False
)


# ==================================================
# SUPPLEMENTARY TABLE S1
# Patient-level primary endpoint results
# ==================================================

patient_frames = []

for tf in TF_ORDER:

    path = (
        ROOT /
        f"Step33_{tf}_MS3_PatientLevel_ProgramDelta.csv"
    )

    df = pd.read_csv(path)

    if "TF" not in df.columns:
        df["TF"] = tf

    df["endpoint"] = (
        df["endpoint"]
        .replace({
            "Core50_LTO_FOXM1":
                "Core50_NetworkCovered",

            "Core50_LTO_MYBL2":
                "Core50_NetworkCovered",

            "Core50_Full":
                "Core50_NetworkCovered_Full_Descriptive"
        })
    )

    df = df.loc[
        df["endpoint"]
        .isin(PRIMARY_ENDPOINTS)
    ].copy()

    patient_frames.append(df)

patient_all = pd.concat(
    patient_frames,
    ignore_index=True
)

patient_all.to_csv(
    TAB_SUPP /
    "TableS1_Step33_PatientLevel_PrimaryEndpointResults.csv",
    index=False
)


# ==================================================
# SUPPLEMENTARY TABLE S2
# Response topology
# ==================================================

topology_frames = []

for tf in TF_ORDER:

    path = (
        ROOT /
        f"Step33_{tf}_Endpoint_ResponseTopology.csv"
    )

    df = pd.read_csv(path)

    if "TF" not in df.columns:
        df["TF"] = tf

    df["endpoint"] = (
        df["endpoint"]
        .replace({
            "Core50_LTO_FOXM1":
                "Core50_NetworkCovered",

            "Core50_LTO_MYBL2":
                "Core50_NetworkCovered",

            "Core50_Full":
                "Core50_NetworkCovered_Full_Descriptive"
        })
    )

    # Normalize gene-number column
    if "genes_after_LTO" not in df.columns:

        if "simulatable_genes" in df.columns:

            df["genes_after_LTO"] = (
                df["simulatable_genes"]
            )

    # Normalize direct-target count
    if "direct_target_overlap_n" not in df.columns:

        if "FOXM1_direct_target_overlap_n" in df.columns:

            df["direct_target_overlap_n"] = (
                df[
                    "FOXM1_direct_target_overlap_n"
                ]
            )

    topology_frames.append(df)

topology_all = pd.concat(
    topology_frames,
    ignore_index=True
)

topology_all.to_csv(
    TAB_SUPP /
    "TableS2_Step33_ResponseTopology.csv",
    index=False
)


# ==================================================
# SUPPLEMENTARY TABLE S3
# Frozen endpoint coverage
# ==================================================

projection_summary = (
    projection
    .groupby(
        [
            "endpoint",
            "status"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

coverage_pub = coverage.merge(
    projection_summary,
    on="endpoint",
    how="left"
)

coverage_pub.to_csv(
    TAB_SUPP /
    "TableS3_Step33_FrozenEndpoint_NetworkCoverage.csv",
    index=False
)


# ==================================================
# SUPPLEMENTARY TABLE S4
# TF eligibility
# ==================================================

eligibility.to_csv(
    TAB_SUPP /
    "TableS4_Step33_FormalTF_Eligibility.csv",
    index=False
)


# ==================================================
# SUPPLEMENTARY TABLE S5
# High-var source audit
# ==================================================

highvar_audit.to_csv(
    TAB_SUPP /
    "TableS5_Step33_HighVarGate_Audit.csv",
    index=False
)


# ==================================================
# SUPPLEMENTARY TABLE S6
# All endpoints
# ==================================================

all_endpoints = pd.read_csv(
    ROOT /
    "Step33_5TF_Integrated_AllEndpoints.csv"
)

all_endpoints.to_csv(
    TAB_SUPP /
    "TableS6_Step33_AllEndpoints.csv",
    index=False
)


# ==================================================
# SUPPLEMENTARY TABLE S7
# Frozen endpoint genes
# ==================================================

frozen_genes = pd.read_csv(
    ROOT /
    "Step33_FROZEN_ENDPOINT_GENESETS.csv"
)

frozen_genes.to_csv(
    TAB_SUPP /
    "TableS7_Step33_FrozenEndpoint_GeneSets.csv",
    index=False
)


print("Main tables:", len(list(TAB_MAIN.glob("*.csv"))))
print(
    "Supplementary tables:",
    len(list(TAB_SUPP.glob("*.csv")))
)

print("=" * 70)
print("Publication tables: COMPLETE")

In [ ]:
# Cell 69 — Main Figure 1: five-TF program heatmap

print("=" * 70)
print("Cell 69 — Main Figure 1")
print("=" * 70)

plot_df = (
    primary
    .pivot(
        index="TF",
        columns="endpoint",
        values="median_delta"
    )
    .reindex(
        index=TF_ORDER,
        columns=PRIMARY_ENDPOINTS
    )
)

consistency_df = (
    primary
    .pivot(
        index="TF",
        columns="endpoint",
        values="direction_consistency"
    )
    .reindex(
        index=TF_ORDER,
        columns=PRIMARY_ENDPOINTS
    )
)

values = plot_df.values.astype(float)

lim = float(
    np.nanmax(
        np.abs(values)
    )
)

fig, ax = plt.subplots(
    figsize=(9.0, 5.5)
)

im = ax.imshow(
    values,
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim,
    aspect="auto"
)

ax.set_xticks(
    np.arange(
        len(PRIMARY_ENDPOINTS)
    )
)

ax.set_xticklabels(
    [
        ENDPOINT_DISPLAY[x]
        for x in PRIMARY_ENDPOINTS
    ],
    rotation=25,
    ha="right"
)

ax.set_yticks(
    np.arange(
        len(TF_ORDER)
    )
)

ax.set_yticklabels(
    TF_ORDER
)

ax.set_xlabel(
    "Frozen MS3 program"
)

ax.set_ylabel(
    "Virtual regulator suppression"
)

ax.set_title(
    "Predicted weakening of MS3 proliferative/mitotic programs"
)

for i, tf in enumerate(TF_ORDER):

    for j, endpoint in enumerate(
        PRIMARY_ENDPOINTS
    ):

        val = plot_df.loc[
            tf,
            endpoint
        ]

        cons = consistency_df.loc[
            tf,
            endpoint
        ]

        ax.text(
            j,
            i,
            f"{val:.3f}\n{cons}",
            ha="center",
            va="center",
            fontsize=8
        )

cbar = fig.colorbar(
    im,
    ax=ax
)

cbar.set_label(
    "Median ΔProgram (KO − baseline)"
)

fig.tight_layout()

MAIN1_PNG = (
    FIG_MAIN /
    "Figure_Main1_Step33_FiveTF_ProgramHeatmap.png"
)

MAIN1_PDF = (
    FIG_MAIN /
    "Figure_Main1_Step33_FiveTF_ProgramHeatmap.pdf"
)

fig.savefig(
    MAIN1_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    MAIN1_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Saved:")
print(MAIN1_PNG)
print(MAIN1_PDF)

In [ ]:
# Cell 70 — Main Figure 2: patient-level consistency

print("=" * 70)
print("Cell 70 — Main Figure 2")
print("=" * 70)

PRIMARY_PATIENTS = [
    "HCC08",
    "HCC09",
    "HCC10"
]

patient_plot = (
    patient_all.loc[
        patient_all["patient"]
        .isin(PRIMARY_PATIENTS)
    ]
    .copy()
)

patient_plot["TF"] = pd.Categorical(
    patient_plot["TF"],
    TF_ORDER,
    ordered=True
)

patient_plot["endpoint"] = pd.Categorical(
    patient_plot["endpoint"],
    PRIMARY_ENDPOINTS,
    ordered=True
)

patient_plot = (
    patient_plot
    .sort_values(
        [
            "TF",
            "endpoint",
            "patient"
        ]
    )
)

row_order = []

for tf in TF_ORDER:

    for ep in PRIMARY_ENDPOINTS:

        row_order.append(
            (tf, ep)
        )

row_lookup = {
    x: i
    for i, x in enumerate(
        row_order
    )
}

patient_offsets = {
    "HCC08": -0.18,
    "HCC09": 0.0,
    "HCC10": 0.18
}

patient_markers = {
    "HCC08": "o",
    "HCC09": "s",
    "HCC10": "^"
}

fig, ax = plt.subplots(
    figsize=(9.5, 11)
)

for patient in PRIMARY_PATIENTS:

    sub = patient_plot.loc[
        patient_plot["patient"]
        == patient
    ]

    y = [
        row_lookup[
            (
                str(r["TF"]),
                str(r["endpoint"])
            )
        ]
        +
        patient_offsets[patient]

        for _, r in sub.iterrows()
    ]

    ax.scatter(
        sub["median_delta"],
        y,
        marker=patient_markers[patient],
        s=35,
        label=patient
    )

ax.axvline(
    0,
    linewidth=1
)

labels = [
    f"{tf} | {ENDPOINT_DISPLAY[ep]}"
    for tf, ep in row_order
]

ax.set_yticks(
    np.arange(
        len(row_order)
    )
)

ax.set_yticklabels(
    labels,
    fontsize=8
)

ax.invert_yaxis()

ax.set_xlabel(
    "Patient-level median ΔProgram (KO − baseline)"
)

ax.set_ylabel(
    "Virtual perturbation | MS3 program"
)

ax.set_title(
    "Cross-patient consistency of predicted MS3 program weakening"
)

ax.legend(
    title="Patient",
    frameon=False
)

fig.tight_layout()

MAIN2_PNG = (
    FIG_MAIN /
    "Figure_Main2_Step33_PatientConsistency.png"
)

MAIN2_PDF = (
    FIG_MAIN /
    "Figure_Main2_Step33_PatientConsistency.pdf"
)

fig.savefig(
    MAIN2_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    MAIN2_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print("Saved:")
print(MAIN2_PNG)
print(MAIN2_PDF)

In [ ]:
# Cell 71 — Supplementary Figures S1–S2

print("=" * 70)
print("Cell 71 — Supplementary Figures")
print("=" * 70)


# ==================================================
# Figure S1
# Response topology
# ==================================================

topo_primary = (
    topology_all.loc[
        topology_all["endpoint"]
        .isin(PRIMARY_ENDPOINTS)
    ]
    .copy()
)

topo_primary[
    "response_fraction"
] = (
    topo_primary[
        "genes_with_nonzero_median_delta"
    ]
    /
    topo_primary[
        "genes_after_LTO"
    ]
)

response_matrix = (
    topo_primary
    .pivot(
        index="TF",
        columns="endpoint",
        values="response_fraction"
    )
    .reindex(
        index=TF_ORDER,
        columns=PRIMARY_ENDPOINTS
    )
)

responsive_n = (
    topo_primary
    .pivot(
        index="TF",
        columns="endpoint",
        values="genes_with_nonzero_median_delta"
    )
    .reindex(
        index=TF_ORDER,
        columns=PRIMARY_ENDPOINTS
    )
)

direct_n = (
    topo_primary
    .pivot(
        index="TF",
        columns="endpoint",
        values="direct_target_overlap_n"
    )
    .reindex(
        index=TF_ORDER,
        columns=PRIMARY_ENDPOINTS
    )
)

fig, ax = plt.subplots(
    figsize=(9, 5.5)
)

im = ax.imshow(
    response_matrix.values,
    vmin=0,
    vmax=1,
    aspect="auto"
)

ax.set_xticks(
    np.arange(
        len(PRIMARY_ENDPOINTS)
    )
)

ax.set_xticklabels(
    [
        ENDPOINT_DISPLAY[x]
        for x in PRIMARY_ENDPOINTS
    ],
    rotation=25,
    ha="right"
)

ax.set_yticks(
    np.arange(
        len(TF_ORDER)
    )
)

ax.set_yticklabels(
    TF_ORDER
)

ax.set_title(
    "Network reach of virtual regulator perturbations"
)

ax.set_xlabel(
    "Frozen MS3 program"
)

ax.set_ylabel(
    "Virtual regulator suppression"
)

for i, tf in enumerate(TF_ORDER):

    for j, ep in enumerate(
        PRIMARY_ENDPOINTS
    ):

        rn = int(
            responsive_n.loc[
                tf,
                ep
            ]
        )

        dn = int(
            direct_n.loc[
                tf,
                ep
            ]
        )

        ax.text(
            j,
            i,
            f"{rn} resp.\n{dn} direct",
            ha="center",
            va="center",
            fontsize=7
        )

cbar = fig.colorbar(
    im,
    ax=ax
)

cbar.set_label(
    "Fraction of network-covered genes with non-zero median Δ"
)

fig.tight_layout()

S1_PNG = (
    FIG_SUPP /
    "Figure_S1_Step33_ResponseTopology.png"
)

S1_PDF = (
    FIG_SUPP /
    "Figure_S1_Step33_ResponseTopology.pdf"
)

fig.savefig(
    S1_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    S1_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ==================================================
# Figure S2
# Frozen endpoint network coverage
# ==================================================

coverage_plot = (
    coverage_pub.loc[
        coverage_pub["endpoint"]
        .isin([
            "Core50_LTO_FOXM1",
            "G2M_Independent",
            "Mitotic_Entry_Independent",
            "Spindle_Chromosome_Independent",
            "Hepatic_Differentiation",
            "Bile_Acid_Metabolism"
        ])
    ]
    .copy()
)

coverage_display = {
    "Core50_LTO_FOXM1":
        "MS3 Core50 (FOXM1 LTO)",

    "G2M_Independent":
        "G2M",

    "Mitotic_Entry_Independent":
        "Mitotic Entry",

    "Spindle_Chromosome_Independent":
        "Spindle / Chromosome",

    "Hepatic_Differentiation":
        "Hepatic Differentiation",

    "Bile_Acid_Metabolism":
        "Bile Acid Metabolism"
}

coverage_plot[
    "display"
] = (
    coverage_plot["endpoint"]
    .map(coverage_display)
)

coverage_plot = (
    coverage_plot
    .sort_values(
        "coverage_fraction",
        ascending=True
    )
)

fig, ax = plt.subplots(
    figsize=(8.5, 5.5)
)

y = np.arange(
    len(coverage_plot)
)

ax.barh(
    y,
    coverage_plot[
        "coverage_fraction"
    ]
)

ax.set_yticks(y)

ax.set_yticklabels(
    coverage_plot["display"]
)

ax.set_xlim(
    0,
    1.05
)

ax.set_xlabel(
    "Fraction represented in frozen CellOracle simulation network"
)

ax.set_title(
    "Coverage of prespecified Step33 transcriptional endpoints"
)

for i, (_, r) in enumerate(
    coverage_plot.iterrows()
):

    ax.text(
        r["coverage_fraction"] + 0.015,
        i,
        f'{int(r["present_n"])}/{int(r["requested_n"])}',
        va="center",
        fontsize=8
    )

fig.tight_layout()

S2_PNG = (
    FIG_SUPP /
    "Figure_S2_Step33_EndpointNetworkCoverage.png"
)

S2_PDF = (
    FIG_SUPP /
    "Figure_S2_Step33_EndpointNetworkCoverage.pdf"
)

fig.savefig(
    S2_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    S2_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


print("Supplementary figures saved:")
print(S1_PNG)
print(S2_PNG)

print("=" * 70)

In [ ]:
# Cell 72 — Legends and article-centered method/result lock

print("=" * 70)
print("Cell 72 — Publication text lock")
print("=" * 70)


# ==================================================
# Figure legends
# ==================================================

figure_legends = """
# Step33 Figure Legends

## Main Figure 1
**Predicted weakening of MS3 proliferative/mitotic programs after virtual regulator suppression.**
Heatmap showing median baseline-anchored ΔProgram values (virtual knockout minus baseline)
for five prespecified MS3-associated regulators across four frozen proliferative/mitotic
programs. Values below zero indicate predicted program weakening. The annotation in each
cell indicates the median ΔProgram and directional consistency across the three primary
biological replication patients (HCC08, HCC09, and HCC10). FOXM1 Core50 scoring uses
leave-target-out to avoid direct inclusion of the perturbed regulator.

## Main Figure 2
**Cross-patient consistency of predicted MS3 program weakening.**
Patient-level median ΔProgram values are shown for HCC08, HCC09, and HCC10 across
the five virtual regulator suppressions and four primary MS3 programs. Patient rather
than cell was treated as the biological replication unit.

## Supplementary Figure S1
**Network reach of virtual regulator perturbations.**
For each regulator-program pair, the heatmap shows the fraction of network-covered
program genes with a non-zero median perturbation response across MS3 cells. Cell
annotations indicate the number of responsive genes and the number directly connected
to the perturbed regulator in the frozen Step29 regulatory network.

## Supplementary Figure S2
**Coverage of prespecified transcriptional endpoints in the CellOracle simulation network.**
Bars indicate the fraction of genes from each frozen transcriptional program represented
in the frozen CellOracle-compatible Step29 network. Genes absent from the simulation
network were not recovered by network reconstruction or post hoc edge addition.
"""

with open(
    REPORT_DIR /
    "Step33_Figure_Legends.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(figure_legends)


# ==================================================
# Table legends
# ==================================================

table_legends = """
# Step33 Table Legends

## Main Table 1
Five-regulator virtual perturbation results for the four primary MS3 proliferative/mitotic programs.

## Main Table 2
Cross-regulator convergence of predicted MS3 program weakening.

## Supplementary Table S1
Patient-level virtual perturbation results for HCC04, HCC07, HCC08, HCC09, and HCC10.
HCC08, HCC09, and HCC10 constitute the primary biological replication set; HCC04 and
HCC07 contain only four MS3 cells each and are retained as sparse-cell sensitivity analyses.

## Supplementary Table S2
Network response topology for each formal regulator and transcriptional endpoint.

## Supplementary Table S3
Coverage of frozen Step33 transcriptional endpoints within the CellOracle-compatible
Step29 regulatory network.

## Supplementary Table S4
Eligibility audit for the five prespecified formal virtual-knockout regulators.

## Supplementary Table S5
High-variable-gene QC audit for E2F8, TFDP1, and E2F1.

## Supplementary Table S6
Integrated results across all primary, secondary, and descriptive Step33 endpoints.

## Supplementary Table S7
Frozen transcriptional endpoint gene definitions used in Step33.
"""

with open(
    REPORT_DIR /
    "Step33_Table_Legends.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(table_legends)


# ==================================================
# Methods / Results / Claim lock
# ==================================================

methods_results = """
# Step33 — Article-Centered Methods and Results Lock

## Scientific question
Which frozen MS3-associated regulators, when virtually suppressed, are predicted to
weaken the conserved proliferative/mitotic transcriptional architecture of MS3 malignant cells?

## Formal regulators
FOXM1, MYBL2, E2F8, TFDP1, and E2F1.

BRCA1 remained a frozen mechanistic candidate and was not used as a formal TF-specific
virtual knockout.

## Regulatory network
The Step29 motif-filtered network was frozen before perturbation analysis.
The provenance network contained 32,530 TF-target edges. CellOracle compatibility required
removal of 140 self-loops, yielding a 32,390-edge simulation topology. No additional edge
was added and no GRN reconstruction or TF reselection was performed.

## CellOracle simulation
Cluster-specific GRNs were fitted across the frozen MS1–MS8 malignant states with ridge
alpha=10. PCA and balanced KNN imputation used 13 PCs, k=301, b_sight=2408, and b_maxl=1204.
Virtual suppression used n_propagation=3, use_randomized_GRN=False, and clip_delta_X=False.

## Perturbation scoring
Program effects were quantified using the Baseline-Anchored CellOracle Perturbation Score.
Gene-wise mean and standard deviation were estimated once from baseline imputed_count
values across the 12,057 malignant cells. Identical baseline parameters were applied to both
baseline and simulated expression matrices.

DeltaProgram = simulated ProgramScore - baseline ProgramScore.

Negative values therefore indicate predicted weakening of the prespecified program.

## Primary biological population
The primary analysis was restricted to the frozen MS3 state (720 cells).

HCC08, HCC09, and HCC10 were treated as the primary biological replication patients.
HCC04 and HCC07 contained only four MS3 cells each and were retained as sparse-cell
sensitivity analyses. Cell-level distributions were descriptive only.

## Primary endpoints
1. Network-covered MS3 Core50
2. G2M Independent
3. Mitotic Entry Independent
4. Spindle / Chromosome Independent

For FOXM1, the Core50 endpoint used leave-target-out to avoid directly scoring the
experimentally perturbed TF itself.

## Secondary directional endpoints
1. Hepatic Differentiation
2. Bile Acid Metabolism

## Main result
Virtual suppression of all five frozen regulators predicted weakening of all four primary
MS3 proliferative/mitotic programs. For every regulator-program pair, the predicted direction
was concordant across HCC08, HCC09, and HCC10.

Across the five regulators, the median ΔProgram values were:
- MS3 Core50: -0.082835
- G2M Independent: -0.064978
- Mitotic Entry Independent: -0.152920
- Spindle / Chromosome Independent: -0.049585

The magnitude and network reach of the predicted perturbation varied across regulators,
whereas the direction of program weakening was highly convergent.

## Negative / boundary result
None of the five virtual perturbations predicted restoration of hepatic differentiation
or bile-acid metabolic programs.

Thus, predicted suppression of the MS3 proliferative/mitotic architecture was not equivalent
to restoration of an MS6-like differentiated/metabolic transcriptional identity.

## Interpretation
The results support a convergent regulatory vulnerability architecture within the frozen
CellOracle model rather than dependence on a single regulator.

## Claim boundary
These results represent in silico counterfactual compatibility within the frozen regulatory
network. They do not establish causal regulation, therapeutic efficacy, experimentally
validated gene essentiality, or lineage conversion.

## Transition to Step34
Step34 will test whether virtual perturbations induce counterfactual displacement within
malignant-cell state space. Program suppression will not be interpreted as state reversal
without explicit state-space evidence.
"""

with open(
    REPORT_DIR /
    "Step33_Methods_Results_ClaimLock.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(methods_results)


# Copy frozen lock + article summary
shutil.copy2(
    ROOT /
    "Step33_FINAL_LOCK.json",
    REPORT_DIR /
    "Step33_FINAL_LOCK.json"
)

shutil.copy2(
    ROOT /
    "Step33_FINAL_ArticleCenteredSummary.csv",
    REPORT_DIR /
    "Step33_FINAL_ArticleCenteredSummary.csv"
)


print("Figure legends: saved")
print("Table legends: saved")
print("Methods/results lock: saved")

print("=" * 70)

In [ ]:
# Cell 73 — Step33 publication package final report

print("=" * 70)
print("Cell 73 — STEP33 PUBLICATION PACKAGE FINAL REPORT")
print("=" * 70)


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


# --------------------------------------------------
# Build manifest
# --------------------------------------------------

manifest_rows = []

for path in sorted(
    PKG.rglob("*")
):

    if not path.is_file():
        continue

    manifest_rows.append({
        "relative_path":
            str(
                path.relative_to(PKG)
            ),

        "size_bytes":
            path.stat().st_size,

        "sha256":
            sha256_file(path)
    })

manifest = pd.DataFrame(
    manifest_rows
)

MANIFEST_PATH = (
    REPORT_DIR /
    "Step33_PUBLICATION_MANIFEST.csv"
)

manifest.to_csv(
    MANIFEST_PATH,
    index=False
)


# --------------------------------------------------
# Count final assets
# --------------------------------------------------

main_png = list(
    FIG_MAIN.glob("*.png")
)

main_pdf = list(
    FIG_MAIN.glob("*.pdf")
)

supp_png = list(
    FIG_SUPP.glob("*.png")
)

supp_pdf = list(
    FIG_SUPP.glob("*.pdf")
)

main_tables = list(
    TAB_MAIN.glob("*.csv")
)

supp_tables = list(
    TAB_SUPP.glob("*.csv")
)


# --------------------------------------------------
# Final package lock
# --------------------------------------------------

PUBLICATION_LOCK = {
    "status":
        "STEP33_PUBLICATION_PACKAGE_FROZEN",

    "analysis_status":
        "STEP33_FROZEN",

    "formal_TFs":
        TF_ORDER,

    "primary_endpoints":
        PRIMARY_ENDPOINTS,

    "primary_replication_patients": [
        "HCC08",
        "HCC09",
        "HCC10"
    ],

    "main_figures":
        len(main_png),

    "supplementary_figures":
        len(supp_png),

    "main_tables":
        len(main_tables),

    "supplementary_tables":
        len(supp_tables),

    "primary_result_rows":
        int(len(primary)),

    "cross_TF_convergence": {
        "all_5_TFs_negative_for_all_4_primary_endpoints":
            True,

        "all_5_TFs_show_3of3_patient_direction_consistency":
            True,

        "hepatic_differentiation_restoration_predicted":
            False,

        "bile_acid_metabolism_restoration_predicted":
            False
    },

    "package_root":
        str(PKG),

    "manifest":
        str(MANIFEST_PATH),

    "next_step":
        "Step34 — Virtual State Transition"
}

LOCK_PATH = (
    REPORT_DIR /
    "Step33_PUBLICATION_PACKAGE_LOCK.json"
)

with open(
    LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        PUBLICATION_LOCK,
        f,
        indent=2,
        ensure_ascii=False
    )


# --------------------------------------------------
# Final report
# --------------------------------------------------

print("Main figures:", len(main_png))
print(
    "Supplementary figures:",
    len(supp_png)
)

print("Main tables:", len(main_tables))
print(
    "Supplementary tables:",
    len(supp_tables)
)

print()
print("Main figure files:")

for x in main_png:
    print(" -", x.name)

print()
print("Supplementary figure files:")

for x in supp_png:
    print(" -", x.name)

print()
print("Main table files:")

for x in main_tables:
    print(" -", x.name)

print()
print("Supplementary table files:")

for x in supp_tables:
    print(" -", x.name)

print()
print("Manifest:")
print(MANIFEST_PATH)

print()
print("Publication lock:")
print(LOCK_PATH)

print()
print("=" * 70)
print("STEP33 ANALYSIS STATUS: FROZEN")
print("STEP33 PUBLICATION PACKAGE: FROZEN")
print("NEXT: STEP34 — VIRTUAL STATE TRANSITION")
print("=" * 70)